# Reproducing Petersen & Buzsaki (2020) from dandiset 000059

Paper. Petersen PC, Buzsaki G (2020). Cooling of medial septum reveals theta phase lag
coordination of hippocampal cell assemblies. *Neuron* 107, 731-744.
[doi:10.1016/j.neuron.2020.05.023](https://doi.org/10.1016/j.neuron.2020.05.023). This attempt used
the full text of the accepted author manuscript in PubMed Central (PMC7442698). The supplementary
PDF could not be obtained, so supplementary figures are tested only where the main text states
their result.

Dandiset. [DANDI:000059](https://dandiarchive.org/dandiset/000059/0.250624.0444), version
0.250624.0444. The paper cites no DANDI version (its data statement points to the lab's website
and CRCNS). This is the most recent published version. Version 0.230907.2101 holds the same 100
files with the same blobs. The first version, 0.210812.1514, held 54 combined files, among them
four sessions without behaviour, and its files have no trials table, which every behavioural
claim here needs; the data check shows this.

Code. The paper's MATLAB code is public (GitHub `petersenpeter/Code_Petersen_Buzsaki_Neuron_2020`,
commit aec2f7e, read for this attempt, not run). It fixes the theta measures, the per-trial
averages, the onset-aligned trial series behind Figures 2G, 2H and 6, and the lists of sessions;
`cooling.py` says which parameter comes from where.

What the dandiset holds. 50 behavioural sessions of five rats on the figure-8 maze, each with a
processed file (trials with cooling state and choice error, position, speed, MS temperature, and
units in 24 sessions) and a raw file (20 kHz wide-band and 1250 Hz LFP). The files carry no cell
types and no place field definitions, and no hippocampal temperature recordings.

The notebook streams every file at the pinned version. One bootstrap draws random numbers; it has
its own seeded generator.

| Claim | Status |
|---|---|
| session-structure, cooling-time-course | tested |
| fig2g-theta-freq-temperature, fig2h-theta-power-temperature, fig2j-frequency-consistent-across-animals, fig2i-theta-pre-cooling-post, disc-theta-frequency-12-percent | tested |
| fig6d-errors-increase, fig6e-errors-recover, fig6c-worst-trials-4-to-8, fig6f-errors-temperature, fig6g-errors-theta-frequency, fig6h-errors-theta-power, fig6i-errors-not-speed | tested |
| fig4e-speed-decreases-within-session, fig5-speed-control-cooling | tested |
| suppl2-place-cell-rates-decrease, disc-interneuron-rates-decrease, suppl2d-interneuron-oscillation-slows, suppl1g-waveforms-unchanged | tested, with putative cell types made here |
| fig4b-fp-flfp-decrease, fig4c-fp-flfp-correlation, fig4ac-fp-flfp-difference-preserved, fig4a-frequency-temperature-slopes, fig4d-precession-slope-decrease, fig4gh-fp-speed | tested, with place fields defined here |
| fig3c-field-size-correlated, fig3d-size-fp-negative, fig3e-model-field-size, fig3f-information-unchanged, fig3hi-population-vector-unchanged | tested, with place fields defined here |
| fig5d-ccg-period, fig5cg-compression, fig5h-time-lags-increase, fig5i-phase-lags-unchanged, fig5j-fraction-active-unchanged, fig5k-spikes-per-cycle-unchanged | tested |
| suppl1-hippocampal-temperature | blocked: the hippocampal temperature recordings are not in the dandiset |

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import time
import warnings
from concurrent.futures import ProcessPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import cooling as C

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

In [ ]:
started = time.time()
S = C.sessions()
with ProcessPoolExecutor(6) as ex:
    beh = dict(zip(S.index, ex.map(C.behaviour_session, S.index)))
unit_keys = [k for k, r in beh.items() if S.loc[k, "in_code_list"] and r["has_state"] and r["n_good_units"] > 0]
with ProcessPoolExecutor(4) as ex:
    units = dict(zip(unit_keys, ex.map(C.unit_session, unit_keys)))

In [ ]:
print(f"dandiset {C.DANDISET_ID} version {C.DANDISET_VERSION}: {len(S)} sessions, "
      f"loaded in {time.time() - started:.0f} s ({len(units)} with unit analyses)")

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
from dandi.dandiapi import DandiAPIClient
with DandiAPIClient() as client:
    vers = {}
    for v in ["0.210812.1514", "0.230907.2101", "0.250624.0444"]:
        assets = list(client.get_dandiset(C.DANDISET_ID, v).get_assets())
        vers[v] = {a.path: (a.size, getattr(a, "blob", None)) for a in assets}
    old_asset = next(a for a in client.get_dandiset(C.DANDISET_ID, "0.210812.1514").get_assets()
                     if "MS10-170319" in a.path)
    old_url = old_asset.get_content_url(follow_redirects=1, strip_query=True)
for v, a in vers.items():
    print(f"{v}: {len(a)} files, {sum(s for s, _ in a.values()) / 1e12:.2f} TB")
same = vers["0.230907.2101"].keys() == vers["0.250624.0444"].keys() and all(
    vers["0.230907.2101"][p] == vers["0.250624.0444"][p] for p in vers["0.250624.0444"])
print("0.230907.2101 and 0.250624.0444 hold the same paths, sizes and blobs:", same)
old_sessions = sorted({C.session_key(p) for p in vers["0.210812.1514"]})
new_sessions = sorted({C.session_key(p) for p in vers["0.250624.0444"]})
print("sessions only in 0.210812.1514:", sorted(set(old_sessions) - set(new_sessions)))
f_old = C.open_h5(old_url)
print("0.210812.1514, file", old_asset.path.split("/")[-1], "has a trials table:", "intervals/trials" in f_old,
      "; has units:", "units" in f_old, "; has LFP:", "processing/ecephys/LFP" in f_old)
f_old.close()

The most recent version and the one before it are the same files. The first version holds four
more sessions (MS12-170712, MS13-171204, -171205, -171206), none with behaviour, and its files
have no trials table, so the pinned version is the only one with the cooling states and the
choice errors.

In [ ]:
rows = []
for k, r in beh.items():
    tr = r.get("trials")
    states = tr.state.value_counts().to_dict() if tr is not None and r["has_state"] else {}
    rows.append({"session": k, "subject": r["subject"], "in_code_list": S.loc[k, "in_code_list"],
                 "fig5_list": S.loc[k, "in_fig5_list"], "code_control": S.loc[k, "code_control"],
                 "trials": r["n_trials"], "has_state": r["has_state"],
                 "pre": states.get("Pre-Cooling", 0), "cool_on": states.get("Cooling on", 0),
                 "cool_off": states.get("Cooling off", 0), "post": states.get("Post-Cooling", 0),
                 "errors": int(tr.error.sum()) if tr is not None else 0,
                 "good_units": r["n_good_units"], "all_units": sum(r["unit_quality_counts"].values()),
                 "frame_ms": round(1000 * r["frame_dt"], 2), "theta_ch": r["theta_channel"], "n_el": r["n_electrodes"],
                 "T_min": round(r["temp_min"], 2), "T_max": round(r["temp_max"], 2), "T_d2_sd": r["temp_d2_sd"],
                 "raw_GB": round(S.loc[k, "raw_size_gb"], 1)})
info = pd.DataFrame(rows).set_index("session")
print("Sessions per subject:", info.groupby("subject").size().to_dict(), "; total", len(info))
print("Paper: 53 sessions in 5 rats (Figure 2). The code's behavioural list (BatchProcessing.m) has",
      sum(len(v) for v in C.CODE_SESSIONS.values()), "sessions;", int(info.in_code_list.sum()), "of them are in the dandiset.")
missing = [f"{a}-{d}" for a, ds in C.CODE_SESSIONS.items() for d in ds if f"{a}-{d}" not in info.index]
print("Code sessions not in the dandiset:", missing)
print("Dandiset sessions not in the code's behavioural list:", info.index[~info.in_code_list].tolist())
print("Code's Figure 5 sessions in the dandiset:", int(info.fig5_list.sum()), "of", len(C.FIG5_SESSIONS))
print("\nTrial-table columns that differ from (condition, cooling state, error):",
      {k: r["trial_columns"] for k, r in beh.items() if set(r["trial_columns"]) != {"condition", "cooling state", "error", "id", "start_time", "stop_time"}})
print("\nUnit qualities over all files:", pd.DataFrame([r["unit_quality_counts"] for r in beh.values()]).sum().astype(int).to_dict())
print("Sessions whose units are all 'unsorted':", [k for k, r in beh.items() if r["unit_quality_counts"] and set(r["unit_quality_counts"]) == {"unsorted"}])
print("\nSubject metadata:", {r["subject"]: (r["subject_meta"].get("age"), r["subject_meta"].get("weight"), r["subject_meta"].get("sex"))
                              for r in beh.values()})
print()
print(info.drop(columns=["T_d2_sd"]).to_string())

The dandiset holds 50 sessions; the paper reports 53 in five rats, and the code's behavioural list
has 53. 47 of the code's sessions are here; six are not (MS12-170712, -170715, MS13-171204, -171205,
-171206, MS21-180806). Three sessions here are not in that list: MS10-170307, and the two sessions of
2018-07-19, which the code's recording list describes as controls without cooling. MS21-180627 has a
trials table without the cooling state, and MS21-180712 has no trials. Units are curated (good,
good2, mua, noise) in 17 sessions; in seven (the six MS10 sessions with units and MS12-170723)
every unit is 'unsorted', so no unit can be selected there. No file has a cell type, a place field, or an
electrode location (every location is 'unknown'). The subject age is a range (P3M/P6M).

In [ ]:
print("Frame interval of position, speed and temperature (ms):", sorted(set(info.frame_ms)),
      "; Methods: OptiTrack at 120 Hz (8.33 ms)")
print("Position, speed and temperature share one timestamp array in every file:", all(r["same_clock"] for r in beh.values()))
print("Speed unit:", {r["speed_unit"] for r in beh.values()}, "; position conversion to metres:", {r["pos_conversion"] for r in beh.values()})
xr = np.array([r["x_range"] for r in beh.values()]); yr = np.array([r["y_range"] for r in beh.values()])
print(f"x range over sessions {xr[:, 0].min():.1f} to {xr[:, 1].max():.1f}, y range {yr[:, 0].min():.1f} to {yr[:, 1].max():.1f} (in cm)")
print("Speed percentiles 1, 50, 99 (cm/s), median over sessions:", np.round(np.median([r["speed_pct"] for r in beh.values()], axis=0), 1).tolist())
lin = pd.DataFrame({k: u["lin"] for k, u in units.items()}).T
print("\nMaze fitted per unit session: stem length L and circle radius R (cm)")
print(lin.round(1).to_string())
print("Diameter 2R:", round(2 * lin.R.min(), 1), "to", round(2 * lin.R.max(), 1), "cm; Methods: 110 cm diameter")

Position is stored in centimetres (conversion 0.01 to metres) and speed in cm/s. The maze circle
fitted to the arm positions has a radius of about 50 cm (diameter about 100 cm against the 110 cm
of the Methods, which may be the outer edge of the track). Position is sampled at 120 Hz in the
2017-2018 MS12 to MS22 sessions and at 40 Hz in the MS10 sessions.

In [ ]:
tshape = pd.DataFrame({k: {"T_min": r["temp_min"], "T_max": r["temp_max"], "unique_values": r["temp_n_unique"],
                           "frac_at_max": r["temp_frac_at_max"], "sd_2nd_diff_1Hz": r["temp_d2_sd"]}
                       for k, r in beh.items()}).T
tshape["code_has_temp_channel"] = [not (k.startswith("MS10") or k in C.CODE_CONTROLS) for k in tshape.index]
print(tshape.round(4).to_string())
grp = tshape.groupby("code_has_temp_channel")
print("\nSD of the second difference of temperature at 1 Hz (C), by whether the code's recording list has a temperature channel:")
print(grp.sd_2nd_diff_1Hz.describe().round(4).to_string())
fig, ax = plt.subplots(1, 3, figsize=(15, 3))
for a, k in zip(ax, ["MS10-170319-180352", "MS21-180719-155941", "MS21-180628-155921"]):
    a.plot(beh[k]["temp_1hz_t"], beh[k]["temp_1hz"], lw=0.8)
    a.set_title(k); a.set_xlabel("time (s)"); a.set_ylabel("MS temperature (C)")
plt.tight_layout(); plt.show()
print("Plotted ranges (C):", {k: (round(beh[k]["temp_1hz"].min(), 2), round(beh[k]["temp_1hz"].max(), 2))
                               for k in ["MS10-170319-180352", "MS21-180719-155941", "MS21-180628-155921"]})

The code's recording list gives no temperature channel for the nine MS10 sessions and for the two
2018-07-19 control sessions (`ch_temp = 0`). In the dandiset these eleven sessions still have a
temperature trace, and it is not a recording. MS10-170307 is a constant 37.0 C. Each of the other
ten runs from exactly 17.0 to 37.0 C and sits at exactly 37.0 C for 16% to 35% of its samples
(frac_at_max), while no recorded trace takes its maximum value more than once (frac_at_max 0).
These are smooth curves of a cooling time course, consistent with a model; their roughness (SD of
the second difference) does not tell them apart from recordings. The trials of the two control
sessions are also labelled with cooling states. Recorded traces contain artefacts up to 50 C in some
sessions (T_max above).

In [ ]:
def state_theta(tr):
    m = tr.groupby("state").theta_freq.mean()
    return {s: round(m.get(s, np.nan), 3) for s in ["Pre-Cooling", "Cooling on", "Cooling off", "Post-Cooling"]}

print("Mean trial theta frequency (Hz) by labelled state, in sessions the code lists as controls or that are not in its list:")
for k in sorted(C.CODE_CONTROLS) + ["MS10-170307-154746"]:
    tr = beh[k]["trials"]
    th = state_theta(tr)
    pre = tr.theta_freq[tr.state == "Pre-Cooling"]; cool = tr.theta_freq[tr.state == "Cooling on"]
    print(f"  {k}: {th}; cooling minus pre {cool.mean() - pre.mean():+.3f} Hz, rank-sum P = {stats.ranksums(cool, pre).pvalue:.2g}")
ms10 = [k for k in info.index if k.startswith("MS10") and info.loc[k, "in_code_list"]]
d = [beh[k]["trials"].groupby("state").theta_freq.mean() for k in ms10]
print("\nMS10 sessions of the code's list: cooling minus pre theta (Hz):", [round(x["Cooling on"] - x["Pre-Cooling"], 2) for x in d])
others = [k for k in info.index if info.loc[k, "in_code_list"] and beh[k]["has_state"] and not k.startswith("MS10")]
d2 = [beh[k]["trials"].groupby("state").theta_freq.mean() for k in others]
print("Other sessions of the code's list: cooling minus pre theta (Hz), median", round(np.median([x["Cooling on"] - x["Pre-Cooling"] for x in d2]), 2),
      ", range", round(min(x["Cooling on"] - x["Pre-Cooling"] for x in d2), 2), "to", round(max(x["Cooling on"] - x["Pre-Cooling"] for x in d2), 2))

In [ ]:
print("Clock check: cooling minus pre theta frequency (Hz) with the theta series shifted in time against the trials.")
sh = pd.DataFrame({k: r["shift_contrast"] for k, r in beh.items() if r.get("shift_contrast")}).T
sel = [k for k in sh.index if info.loc[k, "in_code_list"]]
print("Median over the code's sessions, by shift (s):")
print(sh.loc[sel].median().round(3).to_string())
best = sh.loc[sel].idxmin(axis=1)
print("\nShift giving the most negative contrast, count of sessions:", best.value_counts().sort_index().to_dict())
far = [k for k in sel if best[k] not in (-60, -30, 0, 30, 60)]
print("Sessions where it is more than 60 s from zero:", {k: int(best[k]) for k in far})
print("Controls:", sh.loc[sorted(C.CODE_CONTROLS)].round(3).to_string())

Over the code's sessions the theta slowing is strongest with no shift, and in each session the
best shift is within 120 s of zero (29 at zero); the cooling lasts about ten minutes, so the
contrast changes slowly with the shift. The LFP and the trial table share a clock as far as this
check can tell. In the eight MS10 sessions theta slows by 0.37 to 1.08 Hz in the "Cooling on"
trials, so they were cooled, as the code's hand-noted onsets say, although their temperature is
not a recording. In the two control sessions the "Cooling on" trials differ from the pre trials
by -0.20 and +0.44 Hz, the weakest contrasts but one (MS12-170723, +0.50 Hz, has a recorded
temperature drop) against a median of -0.82 Hz in the other sessions. This is consistent with the
code's description of them as sessions without cooling; it does not prove it.

In [ ]:
uc = pd.concat([u["units"].assign(session=k) for k, u in units.items()], ignore_index=True)
print("Good and good2 units in unit sessions:", len(uc), "; by session:", uc.groupby("session").size().to_dict())
print("\nClock check of spikes against position: spatial information (bits/spike) of every unit's map, true and with spikes shifted by 60 s")
print(uc.groupby("session")[["info_all", "info_all_shift60"]].median().round(3).to_string())
print("\nClock check of spikes against LFP: theta phase locking (mean resultant length) during runs, true and with spikes shifted by 0.5 s")
print(uc.groupby("session")[["theta_mrl", "theta_mrl_shift"]].median().round(3).to_string())

In eleven of the thirteen unit sessions the spatial information of the maps falls 40-fold or more
when the spikes are shifted by 60 s, so spikes and position share a clock as far as this check can
tell. In MS21-180621 and MS22-180621 it does not fall (0.23 against 0.28, and 0.12 against 0.11
bits/spike), and these sessions have 4 and 0 place cells. Their spikes are locked to theta (mean
resultant length 0.37 and 0.34, against 0.11 when shifted), so they belong to the recording; the
outputs cannot say whether the spike and position clocks differ there or the units were not
spatially tuned (both are the rat's first session). The clock of spikes and position could not be
established in these two sessions, and they are left out of the unit claims. Theta phase locking
falls with a 0.5 s shift in every session, so spikes and LFP share a clock. Spikes and the
wide-band data share it too, since mean waveforms at the spike times have the shape of action
potentials (below).

Session sets used below (fixed in the claims file): BEH, the 45 sessions of the code's behavioural
list that are here and have cooling states; TEMP, the 37 BEH sessions with a recorded temperature
(BEH without MS10); UNIT, the 11 BEH sessions with curated units whose spikes carry spatial
information (the 13 with curated units less the two 180621 sessions); FIG5, the 9 UNIT sessions of
the code's Figure 5 list.

In [ ]:
BEH = [k for k in info.index if info.loc[k, "in_code_list"] and info.loc[k, "has_state"]]
TEMP = [k for k in BEH if not k.startswith("MS10")]
NO_CLOCK = ["MS21-180621-141817", "MS22-180621-131028"]
UNIT = [k for k in BEH if k in units and k not in NO_CLOCK]
FIG5 = [k for k in UNIT if S.loc[k, "in_fig5_list"]]
for name, ks in [("BEH", BEH), ("TEMP", TEMP), ("UNIT", UNIT), ("FIG5", FIG5)]:
    print(f"{name}: {len(ks)} sessions;", pd.Series([k.split('-')[0] for k in ks]).value_counts().sort_index().to_dict())
TR = {k: beh[k]["trials"] for k in BEH}

In [ ]:
print("Putative cell types (made here; the files have none):", uc.cell_type.value_counts().to_dict())
print("By session:"); print(pd.crosstab(uc.session, uc.cell_type).to_string())
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].hist(uc.ttp.dropna(), bins=np.arange(0, 1.3, 0.025)); ax[0].axvline(C.NARROW_TTP_MS, color="k", ls="--")
ax[0].set_xlabel("trough-to-peak (ms)"); ax[0].set_ylabel("units")
ax[1].scatter(uc.ttp, uc.rate_trials, s=4); ax[1].set_yscale("log"); ax[1].axhline(C.WIDE_INT_RATE, color="k", ls="--")
ax[1].set_xlabel("trough-to-peak (ms)"); ax[1].set_ylabel("rate in trials (Hz)")
plt.tight_layout(); plt.show()
print("Trough-to-peak range (ms):", uc.ttp.min(), "to", uc.ttp.max(), "; rate range (Hz):", round(uc.rate_trials.min(), 3), "to", round(uc.rate_trials.max(), 1),
      "; units with rate 0 not drawn on the log axis:", int((uc.rate_trials <= 0).sum()))
print("Units without 30 spikes in either waveform window (unclassified):", int((uc.cell_type == "unclassified").sum()))
print("Peak-to-peak amplitude of mean waveforms (uV), median by type:", uc.groupby("cell_type").amp_pre.median().round(1).to_dict())
for thr in [0.35, 0.425, 0.5]:
    t_ = np.where(uc.ttp.isna(), "unclassified", np.where((uc.ttp <= thr) | (uc.rate_trials > C.WIDE_INT_RATE), "interneuron", "pyramidal"))
    print(f"threshold {thr} ms:", pd.Series(t_).value_counts().to_dict())

Trough-to-peak times are bimodal around the 0.425 ms threshold, with a narrow group below it.
Changing the threshold to 0.35 or 0.5 ms moves few units (printed above).

In [ ]:
fields = pd.concat([units[k]["fields"].assign(session=k, subject=units[k]["subject"]) for k in UNIT if len(units[k]["fields"])], ignore_index=True)
print("Place fields of putative pyramidal cells:", len(fields), "in", fields.groupby(["session", "unit"]).ngroups, "cells;",
      "by subject:", fields.groupby("subject").size().to_dict())
print("Pyramidal cells per UNIT session:", uc[(uc.cell_type == "pyramidal") & uc.session.isin(UNIT)].groupby("session").size().to_dict())
print("Place cells per UNIT session:", {k: units[k]["n_place_cells"] for k in UNIT})
print("Field length (cm): median", fields.stop_cm.sub(fields.start_cm).median(), ", range", fields.stop_cm.sub(fields.start_cm).min(), "to", fields.stop_cm.sub(fields.start_cm).max())
print("Fields by segment (0 stem, 1 right arm, 2 left arm):", fields.segment.value_counts().sort_index().to_dict())
print("Paper: n = 365 place fields (Figure 4B); 2948 pairs across 18 sessions (Figure 5C).")

In [ ]:
print("Recordings deposited twice: identical trial start times, temperature traces or trial counts across sessions")
sig_tr = {k: tuple(np.round(r["trials"].start.values[:20], 3)) for k, r in beh.items() if r.get("trials") is not None}
sig_T = {k: (round(float(np.sum(r["temp_1hz"])), 3), len(r["temp_1hz"])) for k, r in beh.items()}
dup_tr = [(a, b) for i, a in enumerate(sig_tr) for b in list(sig_tr)[i + 1:] if sig_tr[a] == sig_tr[b]]
dup_T = [(a, b) for i, a in enumerate(sig_T) for b in list(sig_T)[i + 1:] if sig_T[a] == sig_T[b]]
print("pairs with identical first 20 trial starts:", dup_tr)
print("pairs with identical temperature traces:", dup_T)

No recording appears twice.

## Helpers for the claims

Wilcoxon signed-rank and rank-sum tests are two-sided (scipy, exact where scipy uses it). Each
test prints its n.

In [ ]:
def wsr(a, b=None):
    a = np.asarray(a, float)
    d = a if b is None else a - np.asarray(b, float)
    d = d[np.isfinite(d)]
    if len(d) < 2 or np.all(d == 0):
        return np.nan, len(d)
    return float(stats.wilcoxon(d).pvalue), len(d)

def rks(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    return float(stats.ranksums(a, b).pvalue), len(a), len(b)

def within(x, value, tol):
    return bool(np.isfinite(x) and abs(x - value) <= tol + 1e-12)

def subj(k):
    return k.split("-")[0]

## session-structure

> A session holds about 40 control trials, 20 to 60 cooling trials and 20 to 80 recovery trials,
> 80 to 200 trials in all with a mean of 150.

In [ ]:
counts = pd.DataFrame({k: {"trials": len(t), "pre": (t.state == "Pre-Cooling").sum(), "cooling": (t.state == "Cooling on").sum(),
                           "cooling_off": (t.state == "Cooling off").sum(), "post": (t.state == "Post-Cooling").sum(),
                           "cooling_blocks": int(((t.state == "Cooling on") & (t.state.shift() != "Cooling on")).sum())}
                       for k, t in TR.items()}).T
print(counts.describe().round(1).to_string())
print("\nMean trials per session:", round(counts.trials.mean(), 1), "(expected 150 +- 15); range", counts.trials.min(), "to", counts.trials.max(), "(allowed 72 to 220)")
print("Median pre", counts.pre.median(), "(36 to 44); median cooling", counts.cooling.median(), "(18 to 66); median post", counts.post.median(), "(18 to 88)")
print("Sessions with two cooling blocks:", counts.index[counts.cooling_blocks > 1].tolist())
print("\nBy subject (mean):"); print(counts.groupby(lambda k: subj(k)).mean().round(1).to_string())
ok_mean = within(counts.trials.mean(), 150, 15)
ok_parts = [counts.trials.between(72, 220).all(), 36 <= counts.pre.median() <= 44, 18 <= counts.cooling.median() <= 66, 18 <= counts.post.median() <= 88]
print("\nmean within tolerance:", ok_mean, "; range, pre, cooling, post checks:", ok_parts)

**Result: reproduced.** The 45 BEH sessions hold a mean of 143.8 trials (80 to 204), with
medians of 39 pre-cooling, 47 cooling and 44 post-cooling trials, all within the criterion. 12
sessions have two cooling blocks, which the paper does not mention.

## cooling-time-course

> MS cooling peaked about 60 s after its onset and lasted about 10 to 12 minutes.

In [ ]:
rows = []
for k in TEMP:
    r = beh[k]; t1, T1 = r["temp_1hz_t"], r["temp_1hz"]
    tr = TR[k]
    base = float(np.nanmedian(tr.temperature[tr.state == "Pre-Cooling"]))
    t_first_cool = float(tr.start[tr.state == "Cooling on"].iloc[0])
    after = t1 >= float(tr.start.iloc[0])
    cand = np.flatnonzero(after & (T1 < base - 1) & (t1 > t_first_cool - 600))
    if not len(cand):
        continue
    i0 = cand[0]
    win = (t1 >= t1[i0]) & (t1 <= t1[i0] + 300)
    imin = np.flatnonzero(win)[np.argmin(T1[win])]
    below = np.flatnonzero((t1 > t1[i0]) & (T1 < 32))
    dur32 = np.nan
    if len(below):
        j = below[0]
        back = np.flatnonzero((t1 > t1[j]) & (T1 >= 32))
        dur32 = (t1[back[0]] - t1[j]) / 60 if len(back) else np.nan
    rec = np.flatnonzero((t1 > t1[imin]) & (T1 >= base - 1))
    dur_base = (t1[rec[0]] - t1[i0]) / 60 if len(rec) else np.nan
    rows.append({"session": k, "baseline_C": base, "onset_to_first_cooling_trial_s": t_first_cool - t1[i0],
                 "time_to_min_s": t1[imin] - t1[i0], "min_C": T1[imin], "below32_min": dur32, "to_baseline_min": dur_base})
ct = pd.DataFrame(rows).set_index("session")
print(ct.round(1).to_string())
print("\nTEMP sessions measured:", len(ct), "of", len(TEMP))
print("Median time to minimum:", round(ct.time_to_min_s.median(), 1), "s (expected 60 +- 6); IQR", ct.time_to_min_s.quantile([.25, .75]).round(1).tolist())
print("Median duration below 32 C:", round(ct.below32_min.median(), 2), "min; until within 1 C of baseline:", round(ct.to_baseline_min.median(), 2),
      "min (expected 9 to 13.2)")
print("Median minimum temperature:", round(ct.min_C.median(), 1), "C; range", round(ct.min_C.min(), 1), "to", round(ct.min_C.max(), 1))
print("By subject (median):"); print(ct.groupby(lambda k: subj(k))[["time_to_min_s", "below32_min", "to_baseline_min", "min_C"]].median().round(1).to_string())
for thr in [0.5, 1, 2]:
    vals = []
    for k in TEMP:
        r = beh[k]; t1, T1 = r["temp_1hz_t"], r["temp_1hz"]; tr = TR[k]
        base = float(np.nanmedian(tr.temperature[tr.state == "Pre-Cooling"])); tc = float(tr.start[tr.state == "Cooling on"].iloc[0])
        cand = np.flatnonzero((t1 >= float(tr.start.iloc[0])) & (T1 < base - thr) & (t1 > tc - 600))
        if len(cand):
            w = (t1 >= t1[cand[0]]) & (t1 <= t1[cand[0]] + 300); vals.append(t1[np.flatnonzero(w)[np.argmin(T1[w])]] - t1[cand[0]])
    print(f"onset threshold {thr} C below baseline: median time to minimum {np.median(vals):.1f} s (n = {len(vals)})")

**Result: partially reproduced.** The cooling lasted a median 12.16 minutes from onset
until the temperature was again within 1 C of baseline (reading b, within 9 to 13.2 minutes;
8.77 minutes below 32 C under reading a). The minimum came a median 121 s after the onset (IQR
89 to 138 s), twice the 60 s of the Methods, and the onset threshold (0.5, 1 or 2 C) does not
change this (119 to 122 s).

## fig2g-theta-freq-temperature

> Theta frequency decreased linearly with MS temperature (R = 0.81, P = 5.4e-48).

In [ ]:
AL = {k: C.aligned_series(TR[k], columns=("error", "temperature", "speed", "theta_freq", "theta_power", "theta_freq_10", "theta_power_10")) for k in BEH}

def stack(keys, col, mask_ms10_temp=False):
    rows = []
    for k in keys:
        x = AL[k][col].copy()
        if mask_ms10_temp and col == "temperature" and k.startswith("MS10"):
            x[:] = np.nan
        rows.append(x)
    return np.vstack(rows)

def mean_series(keys, col, mask=False):
    return np.nanmean(stack(keys, col, mask), axis=0)

pos = np.arange(1, C.ALIGN_LENGTH + 1)
readings = {"(a) TEMP sessions": TEMP, "(b) BEH with MS10 curves": BEH}
res_2g = {}
for name, ks in readings.items():
    r, p, n = C.pearson(mean_series(ks, "temperature"), mean_series(ks, "theta_freq"))
    res_2g[name] = (r, p, n)
    print(f"{name}: R = {r:.3f}, P = {p:.2g}, positions = {n}, sessions = {len(ks)}")
r, p, n = C.pearson(mean_series(TEMP, "temperature"), mean_series(BEH, "theta_freq"))
print(f"variant, temperature over TEMP and theta over BEH (as the code with MS10 temperature missing): R = {r:.3f}, P = {p:.2g}, n = {n}")
r, p, n = C.pearson(mean_series(TEMP, "temperature"), mean_series(TEMP, "theta_freq_10"))
print(f"variant, trial theta over frames above 10 cm/s, TEMP: R = {r:.3f}, P = {p:.2g}")
pooled = pd.concat([TR[k] for k in TEMP])
r, p, n = C.pearson(pooled.temperature, pooled.theta_freq)
sl = np.polyfit(pooled.temperature[pooled.theta_freq.notna()], pooled.theta_freq.dropna(), 1)[0]
print(f"variant, single trials pooled over TEMP: R = {r:.3f}, P = {p:.2g}, trials = {n}, slope {sl:.3f} Hz/C")
print("Expected R 0.81 +- 0.081 with P < 1e-10; within under:", [k for k, v in res_2g.items() if within(v[0], 0.81, 0.081) and v[1] < 1e-10])
mt, mf = mean_series(TEMP, "temperature"), mean_series(TEMP, "theta_freq")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(pos, mt, "b"); ax0 = ax[0].twinx(); ax0.plot(pos, mf, "m"); ax[0].set_xlabel("aligned trial position (65 = first cooling trial)")
ax[0].set_ylabel("temperature (C)"); ax0.set_ylabel("theta (Hz)")
ax[1].scatter(mt, mf, s=6); ax[1].set_xlabel("mean temperature (C)"); ax[1].set_ylabel("mean theta frequency (Hz)")
plt.tight_layout(); plt.show()
print("Plotted ranges: temperature", np.nanmin(mt).round(2), "to", np.nanmax(mt).round(2), "C; theta", np.nanmin(mf).round(2), "to", np.nanmax(mf).round(2), "Hz")

**Result: reproduced.** With the code's onset-aligned series, theta frequency correlates
with MS temperature at R = 0.789 (P = 2.8e-43, 198 positions) over the 37 sessions with a
recorded temperature, and at R = 0.758 with the MS10 temperature curves included, both within
0.81 +- 0.081. Over single trials pooled the correlation is R = 0.672, with a slope of 0.065 Hz
per C.

## fig2h-theta-power-temperature

> Theta power also decreased with MS temperature, less tightly than frequency (R = 0.54,
> P = 1.2e-16).

In [ ]:
res_2h = {}
for name, ks in readings.items():
    r, p, n = C.pearson(mean_series(ks, "temperature"), mean_series(ks, "theta_power"))
    res_2h[name] = (r, p, n)
    print(f"{name}: R = {r:.3f}, P = {p:.2g}, positions = {n}; frequency R under the same reading {res_2g[name][0]:.3f}")
r, p, n = C.pearson(mean_series(TEMP, "temperature"), mean_series(TEMP, "theta_power_10"))
print(f"variant, frames above 10 cm/s, TEMP: R = {r:.3f}, P = {p:.2g}")
pooled_p = pd.concat([TR[k].assign(tp=TR[k].theta_power - TR[k].theta_power.mean()) for k in TEMP])
r, p, n = C.pearson(pooled_p.temperature, pooled_p.tp)
print(f"variant, single trials pooled (power minus session mean), TEMP: R = {r:.3f}, P = {p:.2g}, n = {n}")
print("Within tolerance with P < 1e-5 and below the frequency R under:",
      [k for k, v in res_2h.items() if within(v[0], 0.54, 0.054) and v[1] < 1e-5 and v[0] < res_2g[k][0]])

**Result: not reproduced.** Theta power, computed as the code computes it, does not follow
MS temperature: R = -0.056 (P = 0.44) over TEMP sessions and R = 0.123 (P = 0.085) with the MS10
curves, against R = 0.54. Over single trials pooled R = 0.107. The power measure is the code's
(spectrogram magnitude of the band-passed LFP), on the electrode the files flag as theta
reference.

## fig2j-frequency-consistent-across-animals

> The relationship of theta frequency with MS temperature was more consistent across animals
> than that of theta power.

In [ ]:
rows = []
for a in sorted({subj(k) for k in TEMP}):
    ks = [k for k in TEMP if subj(k) == a]
    rf = C.pearson(mean_series(ks, "temperature"), mean_series(ks, "theta_freq"))
    rp = C.pearson(mean_series(ks, "temperature"), mean_series(ks, "theta_power"))
    rows.append({"animal": a, "sessions": len(ks), "R_freq": rf[0], "P_freq": rf[1], "R_power": rp[0], "P_power": rp[1]})
pa = pd.DataFrame(rows).set_index("animal")
print(pa.to_string(float_format=lambda x: f"{x:.3g}"))
print("\nAll frequency R positive with P < 0.05:", bool(((pa.R_freq > 0) & (pa.P_freq < 0.05)).all()))
print("Range of frequency R:", round(pa.R_freq.max() - pa.R_freq.min(), 3), "; range of power R:", round(pa.R_power.max() - pa.R_power.min(), 3))
print("MS10 (no recorded temperature) is not in this table.")

**Result: reproduced.** In each of the four rats with a recorded temperature, theta
frequency correlates positively with temperature (R = 0.685 to 0.954, every P below 1e-22),
a range of 0.269, while the power correlations range over 0.722 (from -0.346 to 0.376). The
outcome rests on power varying across rats, not on a consistent power effect (see
fig2h-theta-power-temperature). MS10 has no recorded temperature and is not in the test.

## fig2i-theta-pre-cooling-post

> Theta frequency and power were lower during MS cooling than before and after it.

In [ ]:
sm = pd.DataFrame({k: {f"{m}_{C.SHORT[s]}": TR[k].loc[TR[k].state == s, m].mean() for m in ["theta_freq", "theta_power"] for s in C.STATES} for k in BEH}).T
print(sm.groupby(lambda k: subj(k)).mean().round(3).to_string())
print("\nMean over sessions:", sm.mean().round(3).to_dict())
out = {}
for m in ["theta_freq", "theta_power"]:
    for other in ["pre", "post"]:
        p, n = wsr(sm[f"{m}_cooling"], sm[f"{m}_{other}"])
        lower = int((sm[f"{m}_cooling"] < sm[f"{m}_{other}"]).sum())
        out[(m, other)] = (p, lower < n - lower)
        print(f"{m}: cooling against {other}, cooling lower in {lower} of {n} sessions, signed-rank P = {p:.2g}, median difference {np.median(sm[f'{m}_cooling'] - sm[f'{m}_{other}']):.3g}")
pn = sm[[c for c in sm.columns if "power" in c]].div(sm[[c for c in sm.columns if "power" in c]].mean(axis=1), axis=0)
print("\nPower relative to the session mean of the three states, mean over sessions:", pn.mean().round(3).to_dict())

**Result: partially reproduced.** Theta frequency is lower during cooling than before
(44 of 45 sessions, P = 1.4e-12) and after (45 of 45, P = 5.7e-14), by medians of 0.81 and 0.80
Hz. Theta
power is lower than before at P = 0.048 (26 of 45 sessions) but not lower than after (19 of 45,
P = 0.94); during cooling it is 0.972 of the session mean of the three states.

## disc-theta-frequency-12-percent

> During MS cooling LFP theta frequency decreased by 12%.

In [ ]:
dec_a = 100 * (1 - sm.theta_freq_cooling / sm.theta_freq_pre)
dec_b = 100 * (1 - sm.theta_freq_cooling / ((sm.theta_freq_pre + sm.theta_freq_post) / 2))
pooled_all = pd.concat([TR[k] for k in BEH])
dec_c = 100 * (1 - pooled_all.theta_freq[pooled_all.state == "Cooling on"].mean() / pooled_all.theta_freq[pooled_all.state == "Pre-Cooling"].mean())
print(f"(a) mean over sessions of the decrease against pre: {dec_a.mean():.2f}% (median {dec_a.median():.2f}, range {dec_a.min():.2f} to {dec_a.max():.2f})")
print(f"(b) against the mean of pre and post: {dec_b.mean():.2f}%")
print(f"(c) pooled trials: {dec_c:.2f}%")
print("By subject, reading (a):", dec_a.groupby(lambda k: subj(k)).mean().round(2).to_dict())
mins = pd.Series({k: 100 * (1 - TR[k].theta_freq[TR[k].state == "Cooling on"].min() / sm.loc[k, "theta_freq_pre"]) for k in BEH})
print(f"variant, decrease at the slowest cooling trial: mean {mins.mean():.2f}%")
print("Within 12 +- 1.2 under:", [n for n, v in [("a", dec_a.mean()), ("b", dec_b.mean()), ("c", dec_c)] if within(v, 12, 1.2)])

**Result: partially reproduced.** Theta frequency decreases during cooling under every
reading, by 8.88% (a), 9.18% (b) and 9.23% (c), all outside 12 +- 1.2%. The slowest cooling trial
of a session is a mean 16.93% below pre. The paper's own within-field values (8.4 and 7.7 Hz) are
matched here (see fig4b-fp-flfp-decrease).

## fig6d-errors-increase

> MS cooling increased choice errors about 3-fold, within each rat (P < 0.05 for all animals) and
> across all animals (P = 1e-6).

In [ ]:
def rates_table(off_as=None, keys=BEH):
    return pd.DataFrame({k: C.error_rates(TR[k], off_as) for k in keys}).T

er = rates_table()
print(er[["pre", "cooling", "post"]].groupby(lambda k: subj(k)).mean().round(2).to_string())
print("\nMean error rate (%) over", len(er), "sessions: pre", round(er.pre.mean(), 2), ", cooling", round(er.cooling.mean(), 2), ", post", round(er.post.mean(), 2))
ratio_a = er.cooling.mean() / er.pre.mean()
ratio_b = er.cooling.mean() / ((er.pre + er.post) / 2).mean()
ok = er.pre > 0
ratio_c = (er.cooling[ok] / er.pre[ok]).median()
print(f"ratio (a) {ratio_a:.2f}; (b) {ratio_b:.2f}; (c) {ratio_c:.2f} over {int(ok.sum())} sessions with pre > 0 (expected 3.0 +- 0.3)")
p_all, n_all = wsr(er.cooling, er.pre)
print(f"cooling against pre over sessions: higher in {(er.cooling > er.pre).sum()} of {n_all}, signed-rank P = {p_all:.2g}")
rat = {}
for a in sorted({subj(k) for k in BEH}):
    e = er.loc[[k for k in BEH if subj(k) == a]]
    p, n = wsr(e.cooling, e.pre)
    rat[a] = (p, n)
    print(f"  {a}: {n} sessions, pre {e.pre.mean():.1f}%, cooling {e.cooling.mean():.1f}%, ratio {e.cooling.mean() / e.pre.mean():.2f}, higher in {(e.cooling > e.pre).sum()}, P = {p:.3g}")
print("Rats with at least 6 sessions and P < 0.05:", [a for a, (p, n) in rat.items() if n >= 6 and p < 0.05], "of", [a for a, (p, n) in rat.items() if n >= 6])
for off in ["cooling", "post"]:
    e2 = rates_table(off)
    print(f"variant, 'Cooling off' trials counted as {off}: pre {e2.pre.mean():.2f}, cooling {e2.cooling.mean():.2f}, post {e2.post.mean():.2f}, ratio (a) {e2.cooling.mean() / e2.pre.mean():.2f}, P = {wsr(e2.cooling, e2.pre)[0]:.2g}")
e3 = rates_table(keys=TEMP)
print(f"variant, TEMP sessions only: ratio (a) {e3.cooling.mean() / e3.pre.mean():.2f}, P = {wsr(e3.cooling, e3.pre)[0]:.2g}")
e4 = pd.DataFrame({k: C.error_rates(beh[k]["trials"]) for k in BEH + sorted(C.CODE_CONTROLS) + ["MS10-170307-154746"]}).T
print(f"variant, with the two controls and MS10-170307: ratio (a) {e4.cooling.mean() / e4.pre.mean():.2f}, P = {wsr(e4.cooling, e4.pre)[0]:.2g}")
print("Controls alone:", e4.loc[sorted(C.CODE_CONTROLS), ["pre", "cooling", "post"]].round(1).to_dict("index"))

**Result: partially reproduced.** Errors rise from 6.67% before cooling to 15.23% during it,
higher in 40 of 45 sessions (P = 3e-07), a ratio of 2.28 (a), 2.31 (b) or 2.53 (c), below
3.0 +- 0.3 under every reading. Within rats the increase is significant for MS10 (P = 0.0078), MS12
(P = 0.039) and MS21 (P = 0.027) but not for MS22 (P = 0.064, 10 of 12 sessions higher); MS13 has 5
sessions, all higher (P = 0.0625, the smallest an exact test allows).

## fig6e-errors-recover

> The error rate after cooling did not differ from the error rate before it.

In [ ]:
p, n = wsr(er.post, er.pre)
print(f"post against pre over {n} sessions: post higher in {(er.post > er.pre).sum()}, lower in {(er.post < er.pre).sum()}, "
      f"signed-rank P = {p:.3g}; median difference {np.median(er.post - er.pre):.2f} points; means {er.pre.mean():.2f} and {er.post.mean():.2f}%")
p2, n2 = wsr(er.cooling, er.post)
print(f"cooling against post: P = {p2:.2g}")

**Result: reproduced.** After cooling the error rate is 6.49% against 6.67% before (median
difference -0.13 points, P = 0.837 over 45 sessions), and it is lower than during cooling
(P = 8e-08).

## fig6c-worst-trials-4-to-8

> The worst memory performance occurred between trials 4 and 8 after the onset of cooling, when MS
> temperature reached its minimum.

In [ ]:
me = mean_series(BEH, "error_smooth")
mt_all = mean_series(TEMP, "temperature")
mf_all = mean_series(BEH, "theta_freq")
onset0 = C.ALIGN_ONSET - 1          # 0-based index of the first cooling trial, which is trial 1 after onset
k_after = np.arange(1, 41)
idx = onset0 + k_after - 1
peak_k = int(k_after[np.nanargmax(me[idx])])
print("Peak of the mean smoothed error series over trials 1 to 40 after onset: trial", peak_k, f"(error rate {np.nanmax(me[idx]):.3f})")
print("Trial of the minimum mean temperature (TEMP):", int(k_after[np.nanargmin(mt_all[idx])]), "; of the minimum mean theta frequency (BEH):", int(k_after[np.nanargmin(mf_all[idx])]))
print("Mean smoothed error rate by trial after onset (1 to 20):", np.round(me[idx][:20], 3).tolist())
print("Mean smoothed error rate over the 40 trials before onset:", round(float(np.nanmean(me[onset0 - 40:onset0])), 3))
raw = mean_series(BEH, "error")
print("Unsmoothed mean error by trial after onset (1 to 20):", np.round(raw[idx][:20], 3).tolist())
for a in sorted({subj(k) for k in BEH}):
    m_a = mean_series([k for k in BEH if subj(k) == a], "error_smooth")
    print(f"  {a}: peak at trial {int(k_after[np.nanargmax(m_a[idx])])}")
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(np.arange(-40, 41), me[onset0 - 41:onset0 + 40], "r"); ax.axvline(0, color="k", lw=0.5)
ax.set_xlabel("trial after onset (1 = first cooling trial)"); ax.set_ylabel("smoothed error rate")
plt.tight_layout(); plt.show()
print("Plotted range:", np.nanmin(me[onset0 - 41:onset0 + 40]).round(3), "to", np.nanmax(me[onset0 - 41:onset0 + 40]).round(3))

**Result: reproduced.** The mean smoothed error rate peaks at trial 7 after the onset of
cooling (0.193, against 0.066 over the 40 trials before), the same trial as the minimum of the mean
temperature and of the mean theta frequency. In single rats the peak of the mean falls later,
at trials 14 to 26.

## fig6f-errors-temperature

> The incidence of errors correlated negatively with MS temperature (R = -0.35, P < 0.01).

In [ ]:
win = slice(9, 160)   # positions 10 to 160, as the code's correlation matrix
def r_err(keys_err, keys_x, col, sl=slice(None)):
    return C.pearson(mean_series(keys_err, "error_smooth")[sl], mean_series(keys_x, col)[sl])
res_6f = {"(a) TEMP, positions 10 to 160": r_err(TEMP, TEMP, "temperature", win),
          "(b) TEMP, all positions": r_err(TEMP, TEMP, "temperature"),
          "(c) BEH with MS10 curves, positions 10 to 160": r_err(BEH, BEH, "temperature", win)}
for k, v in res_6f.items():
    print(f"{k}: R = {v[0]:.3f}, P = {v[1]:.2g}, positions = {v[2]}")
v = r_err(BEH, TEMP, "temperature", win)
print(f"variant, errors over BEH and temperature over TEMP, positions 10 to 160: R = {v[0]:.3f}, P = {v[1]:.2g}")
pooled_e = pd.concat([TR[k] for k in TEMP])
v = C.pearson(pooled_e.error.astype(float), pooled_e.temperature)
print(f"variant, single trials pooled over TEMP (error 0/1): R = {v[0]:.3f}, P = {v[1]:.2g}, n = {v[2]}")
print("Per animal, positions 10 to 160:", {a: tuple(np.round(r_err([k for k in TEMP if subj(k) == a], [k for k in TEMP if subj(k) == a], "temperature", win)[:2], 4))
                                           for a in sorted({subj(k) for k in TEMP})})
print("Within -0.35 +- 0.035 with P < 0.01 under:", [k for k, v in res_6f.items() if within(v[0], -0.35, 0.035) and v[1] < 0.01])

**Result: partially reproduced.** Errors correlate negatively with MS temperature, and more
strongly than reported: R = -0.838 (P = 5.9e-41) over positions 10 to 160, -0.789 over all
positions, and -0.868 with the MS10 curves, none within -0.35 +- 0.035. Over single trials pooled
R = -0.127 (P = 5.5e-20). Per rat R ranges from -0.298 to -0.860.

## fig6g-errors-theta-frequency

> The incidence of errors correlated negatively with theta frequency (R = -0.35, P < 0.01).

In [ ]:
res_6g = {"(a) BEH, positions 10 to 160": r_err(BEH, BEH, "theta_freq", win), "(b) BEH, all positions": r_err(BEH, BEH, "theta_freq")}
for k, v in res_6g.items():
    print(f"{k}: R = {v[0]:.3f}, P = {v[1]:.2g}, positions = {v[2]}")
v = r_err(TEMP, TEMP, "theta_freq", win); print(f"variant, TEMP sessions, positions 10 to 160: R = {v[0]:.3f}, P = {v[1]:.2g}")
v = r_err(BEH, BEH, "theta_freq_10", win); print(f"variant, theta over frames above 10 cm/s: R = {v[0]:.3f}, P = {v[1]:.2g}")
print("Per animal, positions 10 to 160:", {a: tuple(np.round(r_err([k for k in BEH if subj(k) == a], [k for k in BEH if subj(k) == a], "theta_freq", win)[:2], 4))
                                           for a in sorted({subj(k) for k in BEH})})
print("Within -0.35 +- 0.035 with P < 0.01 under:", [k for k, v in res_6g.items() if within(v[0], -0.35, 0.035) and v[1] < 0.01])

**Result: partially reproduced.** Errors correlate negatively with theta frequency at
R = -0.868 (P = 3.9e-47) over positions 10 to 160 and R = -0.733 over all positions, stronger than
-0.35 +- 0.035. Per rat R ranges from -0.341 to -0.829.

## fig6h-errors-theta-power

> The incidence of errors correlated less with theta power; only 2 of 5 animals had a significant
> correlation (R beyond -0.3, P < 0.01).

In [ ]:
rows = []
for a in sorted({subj(k) for k in BEH}):
    ks = [k for k in BEH if subj(k) == a]
    for col in ["theta_power", "theta_freq", "temperature", "speed"]:
        if col == "temperature" and a == "MS10":
            continue
        r, p, n = r_err(ks, ks, col)
        rows.append({"animal": a, "measure": col, "R": r, "P": p, "positions": n})
pa6 = pd.DataFrame(rows)
print(pa6.pivot(index="animal", columns="measure", values="R").round(3).to_string())
print(pa6.pivot(index="animal", columns="measure", values="P").to_string(float_format=lambda x: f"{x:.2g}"))
pw = pa6[pa6.measure == "theta_power"]
n_sig = int(((pw.R < -0.3) & (pw.P < 0.01)).sum())
print("\nAnimals with errors and theta power at R < -0.3 and P < 0.01:", n_sig, pw[(pw.R < -0.3) & (pw.P < 0.01)].animal.tolist())
for thr in [0.0, -0.2, -0.3]:
    print(f"  with R < {thr} and P < 0.01: {int(((pw.R < thr) & (pw.P < 0.01)).sum())}")

**Result: partially reproduced.** One rat (MS10, R = -0.425, P = 1.1e-09) has errors
correlated with theta power beyond -0.3 at P < 0.01, against two in the paper. A second (MS21,
R = -0.202, P = 0.0063) is significant with a weaker correlation, and MS22 is significant with
the opposite sign (R = 0.325).

## fig6i-errors-not-speed

> The incidence of errors did not correlate with the running speed of the rat (R about 0.1,
> P > 0.05 for all animals).

In [ ]:
sp6 = pa6[pa6.measure == "speed"].set_index("animal")
print(sp6[["R", "P", "positions"]].to_string(float_format=lambda x: f"{x:.3g}"))
print("Animals with P < 0.05:", sp6.index[sp6.P < 0.05].tolist())
v = r_err(BEH, BEH, "speed"); print(f"all BEH sessions together: R = {v[0]:.3f}, P = {v[1]:.2g}")
v = r_err(BEH, BEH, "speed", win); print(f"all BEH sessions, positions 10 to 160: R = {v[0]:.3f}, P = {v[1]:.2g}")

**Result: not reproduced.** Errors correlate with running speed in three of five rats at
P < 0.05: MS13 (R = -0.493, P = 8e-11), MS21 (R = 0.216, P = 0.0034) and MS22 (R = 0.247,
P = 0.001); MS10 and MS12 do not (P = 0.32 and 0.49). The signs differ between rats, and over all
sessions together R = 0.093 (P = 0.19).

## fig4e-speed-decreases-within-session

> Running speed decreased relatively monotonically within a session.

In [ ]:
rho = pd.Series({k: stats.spearmanr(np.arange(len(TR[k])), TR[k].speed, nan_policy="omit")[0] for k in BEH})
p, n = wsr(rho)
print(f"Spearman rho of trial speed with trial number: negative in {(rho < 0).sum()} of {len(rho)} sessions ({100 * (rho < 0).mean():.0f}%), "
      f"median {rho.median():.3f}, signed-rank P = {p:.2g}")
print("By subject: negative / sessions:", {a: f"{(rho[[k for k in BEH if subj(k) == a]] < 0).sum()}/{len([k for k in BEH if subj(k) == a])}" for a in sorted({subj(k) for k in BEH})})
ms = mean_series(BEH, "speed")
print("Mean aligned speed (cm/s) in blocks of 20 positions from 41:", [round(float(np.nanmean(ms[i:i + 20])), 1) for i in range(40, 200, 20)])
rho_run = pd.Series({k: stats.spearmanr(np.arange(len(TR[k])), TR[k].speed_run, nan_policy="omit")[0] for k in BEH})
print(f"variant, speed over frames above 10 cm/s: negative in {(rho_run < 0).sum()} of {len(rho_run)}, median {rho_run.median():.3f}")

**Result: reproduced.** Trial speed falls with trial number in 44 of 45 sessions (median
Spearman rho -0.452, P = 4e-13), in every rat, and the aligned mean falls from 86.0 to 73.7 cm/s
over the session.

## fig5-speed-control-cooling

> Mean running speed was 87.3 +- 8.0 cm/s in control trials and 86.2 +- 9.9 cm/s during MS cooling.

In [ ]:
F5B = [k for k in S.index[S.in_fig5_list] if beh[k]["has_state"]]
print("Sessions:", len(F5B), F5B)
for col, name in [("speed", "(a) all frames"), ("speed_run", "(b) frames above 10 cm/s")]:
    ctl = pd.Series({k: beh[k]["trials"].loc[beh[k]["trials"].state.isin(["Pre-Cooling", "Post-Cooling"]), col].mean() for k in F5B})
    cool = pd.Series({k: beh[k]["trials"].loc[beh[k]["trials"].state == "Cooling on", col].mean() for k in F5B})
    print(f"{name}: control {ctl.mean():.1f} +- {ctl.std():.1f}, cooling {cool.mean():.1f} +- {cool.std():.1f} cm/s (mean +- SD over sessions); "
          f"within: {within(ctl.mean(), 87.3, 8.0)}, {within(cool.mean(), 86.2, 9.9)}; cooling minus control P = {wsr(cool, ctl)[0]:.2g}")
    if col == "speed":
        print("   by subject, control:", ctl.groupby(lambda k: subj(k)).mean().round(1).to_dict(), "cooling:", cool.groupby(lambda k: subj(k)).mean().round(1).to_dict())
ctl_b = pd.Series({k: TR[k].loc[TR[k].state.isin(["Pre-Cooling", "Post-Cooling"]), "speed"].mean() for k in BEH})
cool_b = pd.Series({k: TR[k].loc[TR[k].state == "Cooling on", "speed"].mean() for k in BEH})
print(f"variant, all BEH sessions, all frames: control {ctl_b.mean():.1f} +- {ctl_b.std():.1f}, cooling {cool_b.mean():.1f} +- {cool_b.std():.1f}")

**Result: partially reproduced.** Over the 15 sessions of the code's Figure 5 list, the mean
speed in control trials is 79.6 cm/s over frames above 10 cm/s, within 87.3 +- 8.0, but during
cooling it is 76.0 cm/s, just below 86.2 +- 9.9; with all frames both are lower (78.9 and 74.9).
Speed is lower during cooling than in control (P = 0.035 to 0.041), where the paper's values are
close to each other.

## Unit claims: selection

Units of the UNIT sessions typed as above; place fields of putative pyramidal cells by the
automatic criteria of the Methods. The counts below are what each unit claim starts from.

In [ ]:
UC = uc[uc.session.isin(UNIT)].copy()
UC["subject"] = UC.session.map(subj)
F = fields[fields.session.isin(UNIT)].copy()
print("UNIT sessions:", len(UNIT), "; units", len(UC), UC.cell_type.value_counts().to_dict())
print("Place fields:", len(F), "in", F.groupby(["session", "unit"]).ngroups, "place cells; by subject", F.groupby("subject").size().to_dict())
F3 = F[(F.n_pre >= 50) & (F.n_cooling >= 50) & (F.n_post >= 50)].copy()
F3 = F3[F3[["fp_pre", "fp_cooling", "fp_post", "flfp_pre", "flfp_cooling", "flfp_post"]].notna().all(axis=1)]
print("Fields with at least 50 in-field spikes in each of pre, cooling and post:", int(((F.n_pre >= 50) & (F.n_cooling >= 50) & (F.n_post >= 50)).sum()),
      "; of them with an autocorrelogram peak (fp) in all three states:", len(F3))
print("Dropped for spikes:", len(F) - int(((F.n_pre >= 50) & (F.n_cooling >= 50) & (F.n_post >= 50)).sum()), "; for no fp peak:",
      int(((F.n_pre >= 50) & (F.n_cooling >= 50) & (F.n_post >= 50)).sum()) - len(F3))
print("F3 by subject:", F3.groupby("subject").size().to_dict(), "; by session:", F3.groupby("session").size().to_dict())

## suppl2-place-cell-rates-decrease

> Within-field and peak firing rates of place cells decreased moderately during MS cooling (11%
> and 12%; P < 1e-9 and P < 1e-11).

In [ ]:
Fr = F[(F.time_pre > 0) & (F.time_cooling > 0) & (F.infield_rate_pre > 0) & (F.peak_rate_pre > 0)].copy()
print("Fields with time in the field in both states and a pre rate above 0:", len(Fr), "of", len(F))
res_r = {}
for m in ["infield_rate", "peak_rate"]:
    pc = 100 * (1 - Fr[f"{m}_cooling"] / Fr[f"{m}_pre"])
    a, b, c = pc.median(), 100 * (1 - Fr[f"{m}_cooling"].mean() / Fr[f"{m}_pre"].mean()), pc.mean()
    p, n = wsr(Fr[f"{m}_cooling"], Fr[f"{m}_pre"])
    res_r[m] = (a, b, c, p)
    print(f"{m}: pre mean {Fr[m + '_pre'].mean():.2f} Hz, cooling {Fr[m + '_cooling'].mean():.2f} Hz; decrease (a) median {a:.1f}%, (b) ratio of means {b:.1f}%, "
          f"(c) mean {c:.1f}%; signed-rank P = {p:.2g} (n = {n}); lower in {(Fr[m + '_cooling'] < Fr[m + '_pre']).sum()}")
    pp, nn = wsr(Fr[f"{m}_post"], Fr[f"{m}_cooling"])
    print(f"   post against cooling: post mean {Fr[m + '_post'].mean():.2f} Hz, P = {pp:.2g}")
    print("   by subject, reading (b):", {a_: round(100 * (1 - g[f'{m}_cooling'].mean() / g[f'{m}_pre'].mean()), 1) for a_, g in Fr.groupby("subject")})
for mins in [20, 50, 100]:
    g = Fr[(Fr.n_pre >= mins) & (Fr.n_cooling >= mins)]
    print(f"variant, fields with at least {mins} spikes in pre and cooling (n = {len(g)}): within-field decrease (b) {100 * (1 - g.infield_rate_cooling.mean() / g.infield_rate_pre.mean()):.1f}%, "
          f"peak {100 * (1 - g.peak_rate_cooling.mean() / g.peak_rate_pre.mean()):.1f}%")
pyr = UC[UC.cell_type == "pyramidal"]
print(f"variant, all putative pyramidal cells, rate over trial frames above 10 cm/s (n = {len(pyr)}): pre {pyr.rate_pre.mean():.2f}, cooling {pyr.rate_cooling.mean():.2f} Hz, "
      f"decrease {100 * (1 - pyr.rate_cooling.mean() / pyr.rate_pre.mean()):.1f}%, P = {wsr(pyr.rate_cooling, pyr.rate_pre)[0]:.2g}")
print("Readings with both decreases within tolerance (11 +- 1.1, 12 +- 1.2):",
      [r for i, r in enumerate("abc") if within(res_r["infield_rate"][i], 11, 1.1) and within(res_r["peak_rate"][i], 12, 1.2)])

**Result: partially reproduced.** Within-field and peak rates of 636 place fields are lower
during cooling (P = 0.0024 and P = 0.0013), by 7.7% and 6.7% as ratios of means and 3.7% and 5.3%
as medians, short of 11% and 12% and of the P < 0.001 of the criterion. Fields with at least 50
spikes in both states come closer (10.8% and 9.9%). The decrease differs between rats (MS12 rises
by 10.1%).

## disc-interneuron-rates-decrease

> Firing rates of interneurons decreased during MS cooling by about 14%.

In [ ]:
inn = UC[(UC.cell_type == "interneuron") & (UC.rate_pre > 0)]
pc = 100 * (1 - inn.rate_cooling / inn.rate_pre)
p, n = wsr(inn.rate_cooling, inn.rate_pre)
print(f"Putative interneurons: {len(inn)}; rate pre {inn.rate_pre.mean():.2f}, cooling {inn.rate_cooling.mean():.2f}, post {inn.rate_post.mean():.2f} Hz")
print(f"decrease (a) median {pc.median():.1f}%, (b) ratio of means {100 * (1 - inn.rate_cooling.mean() / inn.rate_pre.mean()):.1f}%; lower in {(inn.rate_cooling < inn.rate_pre).sum()}; signed-rank P = {p:.2g} (n = {n})")
print("By subject, reading (b):", {a: round(100 * (1 - g.rate_cooling.mean() / g.rate_pre.mean()), 1) for a, g in inn.groupby("subject")})
for thr in [0.35, 0.5]:
    t_ = (UC.ttp <= thr) | (UC.rate_trials > C.WIDE_INT_RATE)
    g = UC[t_ & UC.ttp.notna() & (UC.rate_pre > 0)]
    print(f"variant, trough-to-peak threshold {thr} ms (n = {len(g)}): decrease (a) {np.median(100 * (1 - g.rate_cooling / g.rate_pre)):.1f}%, "
          f"(b) {100 * (1 - g.rate_cooling.mean() / g.rate_pre.mean()):.1f}%, P = {wsr(g.rate_cooling, g.rate_pre)[0]:.2g}")
nar = UC[(UC.ttp <= C.NARROW_TTP_MS) & (UC.rate_pre > 0)]
print(f"variant, narrow waveform only (n = {len(nar)}): decrease (b) {100 * (1 - nar.rate_cooling.mean() / nar.rate_pre.mean()):.1f}%, P = {wsr(nar.rate_cooling, nar.rate_pre)[0]:.2g}")

**Result: partially reproduced.** Putative interneurons fire less during cooling (127 units,
lower in 75, P = 0.0017), by 6.5% (median) or 9.1% (ratio of means), below 14 +- 1.4%; other
waveform thresholds give 9.4% to 9.9%.

## suppl2d-interneuron-oscillation-slows

> The theta oscillation frequency of putative fast-spiking interneurons decreased during MS
> cooling, commensurate with the LFP theta.

In [ ]:
io = UC[(UC.cell_type == "interneuron")][["session", "subject", "osc_pre", "osc_cooling", "osc_post"]].dropna(subset=["osc_pre", "osc_cooling"])
p, n = wsr(io.osc_cooling, io.osc_pre)
print(f"Interneurons with an autocorrelogram peak in pre and cooling: {len(io)} of {int((UC.cell_type == 'interneuron').sum())}")
print(f"oscillation frequency pre {io.osc_pre.mean():.2f}, cooling {io.osc_cooling.mean():.2f}, post {io.osc_post.mean():.2f} Hz; lower in {(io.osc_cooling < io.osc_pre).sum()}; signed-rank P = {p:.2g} (n = {n})")
print(f"decrease {100 * (1 - io.osc_cooling.mean() / io.osc_pre.mean()):.1f}%; LFP theta decrease in the same sessions (reading (a)) {dec_a[UNIT].mean():.1f}%")
print("By subject:", io.groupby("subject")[["osc_pre", "osc_cooling"]].mean().round(2).to_dict("index"))
nar = UC[(UC.ttp <= C.NARROW_TTP_MS)].dropna(subset=["osc_pre", "osc_cooling"])
print(f"variant, narrow waveform only (n = {len(nar)}): pre {nar.osc_pre.mean():.2f}, cooling {nar.osc_cooling.mean():.2f} Hz, P = {wsr(nar.osc_cooling, nar.osc_pre)[0]:.2g}")

**Result: reproduced.** The oscillation frequency of putative interneurons falls from 9.55
to 8.66 Hz during cooling (lower in 105 of 114, P = 8.7e-16), a decrease of 9.4%, close to the
8.8% decrease of LFP theta in the same sessions.

## suppl1g-waveforms-unchanged

> The waveforms of hippocampal action potentials were not affected by MS cooling.

In [ ]:
wv = UC[(UC.nwf_pre >= 30) & (UC.nwf_cooling >= 30)]
win_t = pd.DataFrame({k: units[k]["waveform_windows"] for k in UNIT}).T
print("Waveform windows (s) and the mean temperature of the coldest cooling trial:"); print(win_t.to_string())
print(f"\nUnits with at least 30 spikes in each window: {len(wv)} of {len(UC)}")
for m in ["ttp", "amp"]:
    p, n = wsr(wv[f"{m}_cooling"], wv[f"{m}_pre"])
    d = wv[f"{m}_cooling"] - wv[f"{m}_pre"]
    print(f"{m}: pre median {wv[m + '_pre'].median():.3f}, cooling median {wv[m + '_cooling'].median():.3f}; median change {d.median():.4f} "
          f"({100 * (d / wv[m + '_pre']).median():.1f}%), mean change {d.mean():.4f}; higher in {(d > 0).sum()}, lower in {(d < 0).sum()}, equal in {(d == 0).sum()}; "
          f"signed-rank P = {p:.2g} (n = {n})")
print("Units whose largest channel moved between windows:", int((wv.chan_pre != wv.chan_cooling).sum()))
for t_ in ["pyramidal", "interneuron"]:
    g = wv[wv.cell_type == t_]
    print(f"  {t_} (n = {len(g)}): ttp P = {wsr(g.ttp_cooling, g.ttp_pre)[0]:.2g}, amplitude change {100 * ((g.amp_cooling - g.amp_pre) / g.amp_pre).median():.1f}%, P = {wsr(g.amp_cooling, g.amp_pre)[0]:.2g}")
for mins in [50, 100]:
    g = UC[(UC.nwf_pre >= mins) & (UC.nwf_cooling >= mins)]
    print(f"variant, at least {mins} spikes (n = {len(g)}): ttp P = {wsr(g.ttp_cooling, g.ttp_pre)[0]:.2g}, amplitude P = {wsr(g.amp_cooling, g.amp_pre)[0]:.2g}, "
          f"median amplitude change {100 * ((g.amp_cooling - g.amp_pre) / g.amp_pre).median():.1f}%")
print("Trough-to-peak is measured on 50 us samples, so a change below one sample (0.05 ms) cannot be seen.")

**Result: not reproduced.** Across 646 units the waveforms differ between the pre window and
the coldest cooling window: trough-to-peak by the signed-rank test (P = 4.9e-10, median change 0,
since most units do not change by a whole 0.05 ms sample) and amplitude by a median 0.8%
(P = 0.00057). Both changes are small, and the two windows are 8 to 32 minutes apart (printed
above), so electrode drift cannot be told apart from cooling here; for interneurons alone neither differs (P = 0.066 and
0.2).

## fig4b-fp-flfp-decrease

> MS cooling decreased both the oscillation frequency of place cells (fp) and the LFP theta
> frequency in their fields (fLFP) by about the same degree, and both recovered (fLFP 8.4, 7.7,
> 8.5 Hz; fp 10.6, 9.5, 10.4 Hz before, during and after; n = 365).

In [ ]:
exp4b = {"flfp_pre": (8.4, 0.38), "flfp_cooling": (7.7, 0.36), "flfp_post": (8.5, 0.29), "fp_pre": (10.6, 0.95), "fp_cooling": (9.5, 0.84), "fp_post": (10.4, 0.72)}
print(f"Fields: {len(F3)}")
for c, (v, tol) in exp4b.items():
    print(f"{c}: mean {F3[c].mean():.2f} +- {F3[c].std():.2f} (expected {v} +- {tol}); within: {within(F3[c].mean(), v, tol)}")
for m in ["fp", "flfp"]:
    p, n = wsr(F3[f"{m}_cooling"], F3[f"{m}_pre"])
    p2, _ = wsr(F3[f"{m}_post"], F3[f"{m}_cooling"])
    print(f"{m}: cooling lower than pre in {(F3[m + '_cooling'] < F3[m + '_pre']).sum()} of {n}, P = {p:.2g}; post higher than cooling P = {p2:.2g}; "
          f"decrease {100 * (1 - F3[m + '_cooling'].mean() / F3[m + '_pre'].mean()):.1f}%")
print("\nBy subject (means):"); print(F3.groupby("subject")[list(exp4b)].mean().round(2).to_string())
print("By session (means):"); print(F3.groupby("session")[list(exp4b)].mean().round(2).to_string())
for mins in [30, 100]:
    g = F[(F.n_pre >= mins) & (F.n_cooling >= mins) & (F.n_post >= mins)].dropna(subset=list(exp4b))
    print(f"variant, at least {mins} spikes per state (n = {len(g)}):", g[list(exp4b)].mean().round(2).to_dict())

**Result: reproduced.** Over 361 place fields (the paper has 365), fLFP is 8.54, 7.77 and
8.60 Hz and fp 10.55, 9.57 and 10.39 Hz before, during and after cooling, all six within the
paper's spreads. Both fall during cooling (fp in 311 fields, P = 3.5e-44; fLFP in 352, P = 1.2e-60)
by 9.2% and 9.0%, and both recover (P = 4.5e-37 and 6.8e-61). Each rat and session shows the same
pattern.

## fig4c-fp-flfp-correlation

> fp and fLFP were correlated across place fields before, during and after cooling (R = 0.61, 0.59
> and 0.36; P < 1e-6 each).

In [ ]:
exp4c = {"pre": (0.61, 0.061), "cooling": (0.59, 0.059), "post": (0.36, 0.036)}
for st, (v, tol) in exp4c.items():
    r, p, n = C.pearson(F3[f"flfp_{st}"], F3[f"fp_{st}"])
    print(f"{st}: R = {r:.3f}, P = {p:.2g}, n = {n} (expected {v} +- {tol}); within: {within(r, v, tol)}")
    print("   within subjects:", {a: round(C.pearson(g[f"flfp_{st}"], g[f"fp_{st}"])[0], 2) for a, g in F3.groupby("subject")})

**Result: partially reproduced.** fp and fLFP correlate positively across fields before
(R = 0.226, P = 1.5e-05), during (R = 0.444) and after cooling (R = 0.157, P = 0.0029), weaker than
0.61, 0.59 and 0.36.

## fig4ac-fp-flfp-difference-preserved

> During MS cooling fp and fLFP decreased in parallel, so the difference fp - fLFP remained constant.

In [ ]:
dpre, dcool, dpost = F3.fp_pre - F3.flfp_pre, F3.fp_cooling - F3.flfp_cooling, F3.fp_post - F3.flfp_post
p, n = wsr(dcool, dpre)
print(f"fp - fLFP: pre {dpre.mean():.3f}, cooling {dcool.mean():.3f}, post {dpost.mean():.3f} Hz (means); median change cooling minus pre {np.median(dcool - dpre):.3f} Hz; "
      f"signed-rank P = {p:.3g} (n = {n})")
print(f"post against cooling: P = {wsr(dpost, dcool)[0]:.2g}")
print("By subject, mean change:", (dcool - dpre).groupby(F3.subject).mean().round(3).to_dict())
r_pre = (F3.fp_pre / F3.flfp_pre).mean(); r_cool = (F3.fp_cooling / F3.flfp_cooling).mean()
print(f"variant, ratio fp / fLFP: pre {r_pre:.3f}, cooling {r_cool:.3f}, P = {wsr(F3.fp_cooling / F3.flfp_cooling, F3.fp_pre / F3.flfp_pre)[0]:.2g}")

**Result: not reproduced.** fp - fLFP falls from 2.009 Hz before cooling to 1.806 Hz during
it (median change -0.128 Hz, P = 0.000132), so the difference is not preserved by the paper's
default test, and it stays at 1.790 Hz after cooling (P = 0.98 against cooling). The ratio fp/fLFP
is preserved (1.237 and 1.232, P = 0.84). The change differs between rats (MS12 -1.03 Hz, MS13 and
MS22 near 0).

## fig4a-frequency-temperature-slopes

> fLFP and fp measured within place fields decreased linearly with MS temperature (fLFP R = 0.66,
> slope 0.071 Hz per C; fp R = 0.38, slope 0.09 Hz per C).

In [ ]:
W = pd.concat([units[k]["windows"].assign(session=k) for k in UNIT if len(units[k]["windows"])], ignore_index=True)
print("20-trial windows with more than 50 in-field spikes:", len(W), "from", W.groupby(["session", "unit", "field"]).ngroups, "fields")
out4a = {}
for m in ["flfp", "fp"]:
    g = W.dropna(subset=[m, "temp"])
    r, p, n = C.pearson(g.temp, g[m])
    slope = np.polyfit(g.temp, g[m], 1)[0]
    out4a[m] = (r, slope, p)
    print(f"{m}: R = {r:.3f}, slope {slope:.4f} Hz/C, P = {p:.2g}, windows = {n}")
print("Expected fLFP R 0.66 +- 0.066 and slope 0.071 +- 0.0071; fp R 0.38 +- 0.038 and slope 0.09 +- 0.009")
print("within:", within(out4a["flfp"][0], 0.66, 0.066), within(out4a["flfp"][1], 0.071, 0.0071), within(out4a["fp"][0], 0.38, 0.038), within(out4a["fp"][1], 0.09, 0.009))
print("Temperature range of windows:", round(W.temp.min(), 1), "to", round(W.temp.max(), 1), "C")
print("By subject (R, slope):", {a: {m: (round(C.pearson(g.dropna(subset=[m]).temp, g.dropna(subset=[m])[m])[0], 3), round(np.polyfit(g.dropna(subset=[m]).temp, g.dropna(subset=[m])[m], 1)[0], 4)) for m in ["flfp", "fp"]}
                                  for a, g in W.groupby(W.session.map(subj))})
gm = W.groupby(["session", "unit", "field"])
wc = W.assign(temp_c=W.temp - gm.temp.transform("mean"), flfp_c=W.flfp - gm.flfp.transform("mean"), fp_c=W.fp - gm.fp.transform("mean"))
for m in ["flfp", "fp"]:
    g = wc.dropna(subset=[m + "_c"])
    print(f"variant, within-field (each field's mean removed), {m}: R = {C.pearson(g.temp_c, g[m + '_c'])[0]:.3f}, slope {np.polyfit(g.temp_c, g[m + '_c'], 1)[0]:.4f}")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for a, m in zip(ax, ["flfp", "fp"]):
    g = W.dropna(subset=[m]); a.scatter(g.temp, g[m], s=1, alpha=0.2); a.set_xlabel("temperature at spikes (C)"); a.set_ylabel(m + " (Hz)")
plt.tight_layout(); plt.show()
print("Plotted ranges: fLFP", round(W.flfp.min(), 2), "to", round(W.flfp.max(), 2), "Hz; fp", round(W.fp.min(), 2), "to", round(W.fp.max(), 2), "Hz")

**Result: partially reproduced.** Over 52035 windows of 20 trials, fLFP rises with
temperature at R = 0.677 and 0.0714 Hz per C, and fp at a slope of 0.0859 Hz per C, all three
within tolerance; the fp correlation is R = 0.338, just below 0.38 +- 0.038.

## fig3c-field-size-correlated

> Place field size was highly correlated before and during MS cooling (R = 0.91).

In [ ]:
fs = F.dropna(subset=["size_pre", "size_cooling"])
fs = fs[(fs.n_pre > 0) & (fs.n_cooling > 0)]
r, p, n = C.pearson(fs.size_pre, fs.size_cooling)
print(f"Fields with spikes in both states: {n}; R = {r:.4f}, P = {p:.2g} (expected 0.91 +- 0.091, so 0.819 to 1); within: {within(r, 0.91, 0.091)}")
print(f"mean size pre {fs.size_pre.mean():.1f}, cooling {fs.size_cooling.mean():.1f} cm; signed-rank P = {wsr(fs.size_cooling, fs.size_pre)[0]:.2g}")
for mins in [20, 50, 100]:
    g = fs[(fs.n_pre >= mins) & (fs.n_cooling >= mins)]
    print(f"variant, at least {mins} spikes in each state (n = {len(g)}): R = {C.pearson(g.size_pre, g.size_cooling)[0]:.4f}")
print("By subject:", {a: round(C.pearson(g.size_pre, g.size_cooling)[0], 3) for a, g in fs.groupby("subject")})
r2, _, _ = C.pearson(fs.size_pre, fs.size_post); print(f"pre against post: R = {r2:.3f}")

**Result: reproduced.** Field sizes before and during cooling correlate at R = 0.8197 over
635 fields, at the edge of 0.91 +- 0.091 (0.819); with at least 20 or 50 spikes per state R is
0.8348 and 0.8308, and with at least 100 it is 0.8136, outside. Sizes do not differ between the
states (62.7 and 64.9 cm, P = 0.084).

## fig3d-size-fp-negative

> Place field size and fp were negatively correlated before, during and after cooling (R = -0.49,
> -0.40, -0.40; P < 1e-9).

In [ ]:
exp3d = {"pre": (-0.49, 0.049), "cooling": (-0.40, 0.04), "post": (-0.40, 0.04)}
for st, (v, tol) in exp3d.items():
    r, p, n = C.pearson(F3[f"size_{st}"], F3[f"fp_{st}"])
    print(f"{st}: R = {r:.3f}, P = {p:.2g}, n = {n} (expected {v} +- {tol}); within: {within(r, v, tol)}")
    print("   by subject:", {a: round(C.pearson(g[f"size_{st}"], g[f"fp_{st}"])[0], 2) for a, g in F3.groupby("subject")})
for st in exp3d:
    print(f"variant, Spearman, {st}: rho = {stats.spearmanr(F3[f'size_{st}'], F3[f'fp_{st}'])[0]:.3f}")

**Result: partially reproduced.** Field size and fp correlate negatively before, during and
after cooling (R = -0.311, -0.307 and -0.323, every P below 3e-09), weaker than -0.49 and -0.40.

## fig3e-model-field-size

> Measured place field size correlated with the size predicted by SPF = v / (fp - fLFP) before
> (R = 0.55) and during (R = 0.54) cooling, and weakly after (R = 0.11).

In [ ]:
exp3e = {"pre": (0.55, 0.055), "cooling": (0.54, 0.054), "post": (0.11, 0.011)}
for st, (v, tol) in exp3e.items():
    dfq = F3[f"fp_{st}"] - F3[f"flfp_{st}"]
    ok = dfq > 0.2
    pred = F3.loc[ok, f"v_{st}"] / dfq[ok]
    r, p, n = C.pearson(F3.loc[ok, f"size_{st}"], pred)
    print(f"{st}: fields {n} (of {len(F3)}), R = {r:.3f}, P = {p:.2g} (expected {v} +- {tol}); within: {within(r, v, tol)}; "
          f"median predicted {pred.median():.1f} cm, measured {F3.loc[ok, f'size_{st}'].median():.1f} cm")
    rs = stats.spearmanr(F3.loc[ok, f"size_{st}"], pred)[0]
    print(f"   variant, Spearman rho = {rs:.3f}; by subject R:", {a: round(C.pearson(F3.loc[ok & (F3.subject == a), f"size_{st}"], pred[F3.loc[ok, 'subject'] == a])[0], 2) for a in sorted(F3.subject.unique())})

**Result: partially reproduced.** Measured sizes correlate with v / (fp - fLFP) before
(R = 0.246) and during cooling (R = 0.344), both with P below 1e-5 but below 0.55 and 0.54, and the
post value (R = 0.325) is far above the paper's 0.11.

## fig3f-information-unchanged

> The information content of place cell spikes was not altered by MS cooling (P > 0.26).

In [ ]:
pcells = F[["session", "unit"]].drop_duplicates()
pi = UC.merge(pcells, on=["session", "unit"])
print("Place cells:", len(pi))
for a, b in [("pre", "cooling"), ("cooling", "post"), ("pre", "post")]:
    p, n = wsr(pi[f"info_{b}"], pi[f"info_{a}"])
    print(f"{a} against {b}: medians {pi['info_' + a].median():.3f} and {pi['info_' + b].median():.3f} bits/spike; median change {np.nanmedian(pi['info_' + b] - pi['info_' + a]):.3f}; "
          f"signed-rank P = {p:.2g} (n = {n})")
print("By subject, median pre and cooling:", pi.groupby("subject")[["info_pre", "info_cooling"]].median().round(3).to_dict("index"))

**Result: not reproduced.** The paper's default paired test finds information content lower
during cooling than before (median change -0.029 against a median of 1.253 bits/spike, P = 0.014
over 374 place cells) and lower after cooling than before (P = 0.0064); cooling and post do not
differ (P = 0.31).

## fig3hi-population-vector-unchanged

> The spatial scale of the population vector correlation of pyramidal cells was not affected by MS
> cooling.

In [ ]:
pvt = pd.DataFrame({k: {"pre": units[k]["pv"]["pre"], "cooling": units[k]["pv"]["cooling"], "post": units[k]["pv"]["post"],
                        "pyramidal": int(((UC.session == k) & (UC.cell_type == "pyramidal")).sum())} for k in UNIT}).T
print("Distance (cm) at which the population vector correlation drops below 0.5:"); print(pvt.round(2).to_string())
p, n = wsr(pvt.cooling.astype(float), pvt.pre.astype(float))
print(f"cooling against pre: median change {np.nanmedian(pvt.cooling.astype(float) - pvt.pre.astype(float)):.2f} cm, signed-rank P = {p:.3g} (n = {n})")
print(f"cooling against post: P = {wsr(pvt.cooling.astype(float), pvt.post.astype(float))[0]:.3g}")

**Result: reproduced.** The distance at which the population vector correlation falls below
0.5 does not change with cooling (median change 0.25 cm, P = 0.966 over 11 sessions; 18.3 to 36.9
cm across sessions and states).

## fig4d-precession-slope-decrease

> The phase precession slope decreased slightly but significantly during MS cooling (P < 0.007 and
> P < 0.0003).

In [ ]:
fsl = F3.dropna(subset=["slope_pre", "slope_cooling", "slope_post"])
print(f"Fields with a slope in every state: {len(fsl)} of {len(F3)}; slopes in rad/cm")
print("Median slope pre, cooling, post:", fsl[["slope_pre", "slope_cooling", "slope_post"]].median().round(4).to_dict(),
      "; fraction negative in pre:", round(float((fsl.slope_pre < 0).mean()), 3))
for name, f_ in [("(a) magnitude", np.abs), ("(b) signed", lambda x: x)]:
    p1, n1 = wsr(f_(fsl.slope_cooling), f_(fsl.slope_pre))
    p2, n2 = wsr(f_(fsl.slope_cooling), f_(fsl.slope_post))
    d1 = np.median(f_(fsl.slope_cooling) - f_(fsl.slope_pre)); d2 = np.median(f_(fsl.slope_cooling) - f_(fsl.slope_post))
    print(f"{name}: cooling minus pre median {d1:.4f}, P = {p1:.2g}; cooling minus post median {d2:.4f}, P = {p2:.2g} (n = {n1})")
neg = fsl[(fsl.slope_pre < 0) & (fsl.slope_cooling < 0) & (fsl.slope_post < 0)]
print(f"variant, fields precessing (negative slope) in all three states (n = {len(neg)}): magnitude cooling minus pre median {np.median(neg.slope_cooling.abs() - neg.slope_pre.abs()):.4f}, "
      f"P = {wsr(neg.slope_cooling.abs(), neg.slope_pre.abs())[0]:.2g}; against post P = {wsr(neg.slope_cooling.abs(), neg.slope_post.abs())[0]:.2g}")
print("Mean resultant length of the fits, median:", fsl[["rbar_pre", "rbar_cooling", "rbar_post"]].median().round(3).to_dict())

**Result: partially reproduced.** Over 361 fields, the magnitude of the precession slope
during cooling does not differ from pre (median change 0, P = 0.83) and is smaller than post
(median -0.002 rad/cm, P = 0.022), so under reading (a) one comparison holds; the signed slope
does not differ in either (P = 0.32 and 0.15).

## fig4gh-fp-speed

> fp depended on running speed more strongly than fLFP, and fp during cooling was lower at all
> velocities.

In [ ]:
rng_4gh = np.random.default_rng(4)
bins = list(range(20, 120, 20))
long = []
for _, r_ in F.iterrows():
    for st in ["pre", "cooling", "post"]:
        for b in bins:
            fp_, fl_ = r_[f"fp_{st}_v{b}"], r_[f"flfp_{st}_v{b}"]
            if np.isfinite(fp_) and np.isfinite(fl_):
                long.append({"field": (r_.session, r_.unit, r_.field), "state": st, "speed": b + 10, "fp": fp_, "flfp": fl_})
L = pd.DataFrame(long)
print("Field, state and speed-bin values:", len(L), "from", L.field.nunique(), "fields; per bin:", L.groupby("speed").size().to_dict())
s_fp = stats.linregress(L.speed, L.fp); s_fl = stats.linregress(L.speed, L.flfp)
print(f"fp slope {s_fp.slope:.4f} Hz per cm/s (P = {s_fp.pvalue:.2g}); fLFP slope {s_fl.slope:.4f} (P = {s_fl.pvalue:.2g})")
codes = pd.factorize(L.field)[0]
rows_by = [np.flatnonzero(codes == i) for i in range(codes.max() + 1)]
boot = []
for _ in range(1000):
    pick = rng_4gh.integers(0, len(rows_by), len(rows_by))
    ii = np.concatenate([rows_by[i] for i in pick])
    x = L.speed.values[ii]
    boot.append(np.polyfit(x, L.fp.values[ii], 1)[0] - np.polyfit(x, L.flfp.values[ii], 1)[0])
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"difference of slopes (fp minus fLFP) {s_fp.slope - s_fl.slope:.4f}, 95% bootstrap interval over fields {lo:.4f} to {hi:.4f}")
print("\nMedian fp by speed bin, fields measured in both pre and cooling:")
part2 = []
for b in bins:
    pr = L[(L.state == "pre") & (L.speed == b + 10)].set_index("field").fp
    co = L[(L.state == "cooling") & (L.speed == b + 10)].set_index("field").fp
    both = pr.index.intersection(co.index)
    if len(both) >= 10:
        part2.append(co[both].median() < pr[both].median())
        print(f"  {b}-{b + 20} cm/s: n = {len(both)}, pre {pr[both].median():.2f}, cooling {co[both].median():.2f} Hz, P = {wsr(co[both], pr[both])[0]:.2g}")
    else:
        print(f"  {b}-{b + 20} cm/s: n = {len(both)} (fewer than 10)")
print("fp lower during cooling in every bin with at least 10 fields:", all(part2), f"({sum(part2)} of {len(part2)})")
print("Mean fp and fLFP by speed bin (all states):", L.groupby("speed")[["fp", "flfp"]].mean().round(2).to_dict("index"))

**Result: reproduced.** fp rises with running speed at 0.0136 Hz per cm/s (P = 3.2e-12),
more than fLFP at 0.0061 (difference 0.0074, bootstrap interval 0.0036 to 0.0111), and in the
three speed bins with at least 10 fields fp is lower during cooling (medians 8.93 to 9.52
against 9.95 to 10.31 Hz, P below 1e-5 each). Below 60 cm/s too few fields could be measured.

## fig5d-ccg-period

> The theta-repeated cross-correlograms of place cell pairs had a period of about 85 ms before
> cooling, which lengthened during MS cooling (P = 0.002), while the period of the phase
> cross-correlograms was unaffected (P = 0.32).

In [ ]:
P5 = pd.concat([units[k]["pairs"].assign(session=k) for k in FIG5 if len(units[k]["pairs"])], ignore_index=True)
print("FIG5 sessions:", len(FIG5), "; pairs of overlapping fields:", len(P5), "; by session:", P5.groupby("session").size().to_dict())
print("Pairs with a main peak (time) in pre, cooling, post:", {c: int(P5[f"tau_{c}"].notna().sum()) for c in ["pre", "cooling", "post"]},
      "; with at least 50 spikes of each cell:", {c: int(((P5[f"na_{c}"] >= 50) & (P5[f"nb_{c}"] >= 50)).sum()) for c in ["pre", "cooling", "post"]})
rows = []
for k, g in P5.groupby("session"):
    ctl_t = pd.concat([g.period_pre, g.period_post]); ctl_p = pd.concat([g.phase_period_pre, g.phase_period_post])
    rows.append({"session": k, "control_ms": ctl_t.median(), "cooling_ms": g.period_cooling.median(), "pre_ms": g.period_pre.median(), "post_ms": g.period_post.median(),
                 "control_rad_pi": ctl_p.median() / np.pi, "cooling_rad_pi": g.phase_period_cooling.median() / np.pi,
                 "n_control": int(ctl_t.notna().sum()), "n_cooling": int(g.period_cooling.notna().sum())})
per = pd.DataFrame(rows).set_index("session")
print(per.round(2).to_string())
print(f"\nMedian control period over sessions: {per.control_ms.median():.1f} ms (expected 85 +- 8.5); cooling {per.cooling_ms.median():.1f} ms")
p_t, n_t = wsr(per.cooling_ms, per.control_ms); p_p, n_p = wsr(per.cooling_rad_pi, per.control_rad_pi)
print(f"cooling longer than control in {(per.cooling_ms > per.control_ms).sum()} of {n_t} sessions, signed-rank P = {p_t:.3g}")
print(f"phase period: control {per.control_rad_pi.median():.3f} pi, cooling {per.cooling_rad_pi.median():.3f} pi rad; longer in {(per.cooling_rad_pi > per.control_rad_pi).sum()} of {n_p}, P = {p_p:.3g}")
print("LFP theta period in the same sessions (1000 / mean trial theta), control and cooling (ms):",
      round(float(np.median([1000 / ((sm.loc[k, 'theta_freq_pre'] + sm.loc[k, 'theta_freq_post']) / 2) for k in FIG5])), 1),
      round(float(np.median([1000 / sm.loc[k, 'theta_freq_cooling'] for k in FIG5])), 1))
print("Pooled pairs, median period: pre", round(P5.period_pre.median(), 1), ", cooling", round(P5.period_cooling.median(), 1), ", post", round(P5.period_post.median(), 1), "ms")
# variant: period of the summed correlogram of all pairs, by spectral peak and by the fit
from scipy.ndimage import gaussian_filter1d
for k in FIG5:
    cs = units[k]["ccg_sum"]
    ctl = cs["pre"]["time"] + cs["post"]["time"]
    sp_ = []
    for c in [ctl, cs["cooling"]["time"]]:
        res = c - gaussian_filter1d(c, 40, mode="nearest")
        sel = np.abs(C.TIME_LAGS) <= 300
        z = res[sel] * np.hanning(sel.sum()); spec = np.abs(np.fft.rfft(z, 1 << 16)); fr = np.fft.rfftfreq(1 << 16, 0.001)
        band = (fr >= 5) & (fr <= 14); sp_.append(1000 / fr[band][np.argmax(spec[band])])
    print(f"variant, summed correlogram spectral period {k}: control {sp_[0]:.1f}, cooling {sp_[1]:.1f} ms")

**Result: partially reproduced.** The period of the pair cross-correlograms lengthens during
cooling, from a median 109.2 to 114.0 ms (longer in 8 of 9 sessions, P = 0.0195), and the phase
period does not change (1.740 and 1.760 pi rad, P = 0.59). The control period is 109.2 ms, not
85 +- 8.5 ms; it is shorter than the LFP theta period (115.7 ms), as the paper says of its 85 ms.

## fig5cg-compression

> Distance-time compression of place cell pairs (0.54 cm/ms before cooling) decreased during MS
> cooling (P = 0.02), while distance-phase compression was unchanged (P = 0.92).

In [ ]:
def slope_xy(x, y):
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < 5:
        return np.nan
    return float(np.polyfit(x[ok], y[ok], 1)[0])

c_pre = slope_xy(P5.tau_pre.values, P5.ltr.values)
print(f"Compression over pooled pre pairs (slope of Ltr on tau): {c_pre:.3f} cm/ms (expected 0.54 +- 0.054), pairs = {int(P5.tau_pre.notna().sum())}")
print(f"  variant, inverse slope of tau on Ltr: {1 / slope_xy(P5.ltr.values, P5.tau_pre.values):.3f} cm/ms")
print(f"  variant, half-period window: {slope_xy(P5.tau_half_pre.values, P5.ltr.values):.3f} cm/ms")
r_, p_, n_ = C.pearson(P5.tau_pre, P5.ltr); print(f"  correlation of tau and Ltr in pre: R = {r_:.3f}, P = {p_:.2g}")
rows = []
for k, g in P5.groupby("session"):
    t_ctl = np.r_[g.tau_pre.values, g.tau_post.values]; l_ctl = np.r_[g.ltr.values, g.ltr.values]
    ph_ctl = np.r_[g.phase_pre.values, g.phase_post.values]
    rows.append({"session": k, "c_control": slope_xy(t_ctl, l_ctl), "c_cooling": slope_xy(g.tau_cooling.values, g.ltr.values),
                 "c_pre": slope_xy(g.tau_pre.values, g.ltr.values),
                 "cphase_control": slope_xy(ph_ctl, l_ctl), "cphase_cooling": slope_xy(g.phase_cooling.values, g.ltr.values),
                 "inv_control": 1 / slope_xy(l_ctl, t_ctl), "inv_cooling": 1 / slope_xy(g.ltr.values, g.tau_cooling.values),
                 "half_control": slope_xy(np.r_[g.tau_half_pre.values, g.tau_half_post.values], l_ctl), "half_cooling": slope_xy(g.tau_half_cooling.values, g.ltr.values)})
cmp_ = pd.DataFrame(rows).set_index("session")
print(cmp_.round(3).to_string())
for a, b, name in [("c_cooling", "c_control", "time compression (cm/ms)"), ("cphase_cooling", "cphase_control", "phase compression (cm/rad)"),
                   ("inv_cooling", "inv_control", "variant, inverse slope (cm/ms)"), ("half_cooling", "half_control", "variant, half-period window (cm/ms)")]:
    p, n = wsr(cmp_[a], cmp_[b])
    print(f"{name}: control median {cmp_[b].median():.3f}, cooling {cmp_[a].median():.3f}; cooling lower in {(cmp_[a] < cmp_[b]).sum()} of {n}, signed-rank P = {p:.3g}")

**Result: not reproduced.** The distance-time compression over pooled pre pairs is
0.273 cm/ms, half of 0.54, and the per-session compression is lower during cooling in 8 of 9
sessions without reaching significance (0.298 against 0.191 cm/ms, P = 0.0547). The phase
compression does not change (P = 0.426). Time lag and distance correlate weakly in these pairs
(R = 0.195), so the slope depends on the fit: the inverse regression gives 9.085 cm/ms.

## fig5h-time-lags-increase

> During MS cooling the theta time lags between place cell pairs increased compared with before and
> after cooling (P = 8e-5 and P = 0.00032), and did not differ between before and after (P = 0.79).

In [ ]:
for a, b in [("cooling", "pre"), ("cooling", "post"), ("post", "pre")]:
    x, y = P5[f"tau_{a}"].abs(), P5[f"tau_{b}"].abs()
    ok = x.notna() & y.notna()
    p, n = wsr(x[ok], y[ok])
    print(f"(a) pairs, |tau| {a} against {b}: medians {x[ok].median():.1f} and {y[ok].median():.1f} ms, mean difference {(x[ok] - y[ok]).mean():.2f} ms, "
          f"{a} longer in {(x[ok] > y[ok]).sum()}, shorter in {(x[ok] < y[ok]).sum()}; signed-rank P = {p:.3g} (n = {n})")
ses = P5.assign(**{f"abs_{c}": P5[f"tau_{c}"].abs() for c in ["pre", "cooling", "post"]}).groupby("session")[["abs_pre", "abs_cooling", "abs_post"]].median()
print(ses.round(1).to_string())
for a, b in [("cooling", "pre"), ("cooling", "post"), ("post", "pre")]:
    p, n = wsr(ses[f"abs_{a}"], ses[f"abs_{b}"])
    print(f"(b) sessions, median |tau| {a} against {b}: longer in {(ses['abs_' + a] > ses['abs_' + b]).sum()} of {n}, P = {p:.3g}")
for a, b in [("cooling", "pre"), ("cooling", "post")]:
    x, y = P5[f"tau_half_{a}"].abs(), P5[f"tau_half_{b}"].abs(); ok = x.notna() & y.notna()
    print(f"variant, half-period window, {a} against {b}: medians {x[ok].median():.1f} and {y[ok].median():.1f}, P = {wsr(x[ok], y[ok])[0]:.3g} (n = {int(ok.sum())})")
PU = pd.concat([units[k]["pairs"].assign(session=k) for k in UNIT if len(units[k]["pairs"])], ignore_index=True)
x, y = PU.tau_cooling.abs(), PU.tau_pre.abs(); ok = x.notna() & y.notna()
print(f"variant, all UNIT sessions ({PU.session.nunique()}), cooling against pre: medians {x[ok].median():.1f} and {y[ok].median():.1f}, P = {wsr(x[ok], y[ok])[0]:.3g} (n = {int(ok.sum())})")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for c, col in [("pre", "r"), ("cooling", "b"), ("post", "g")]:
    v = np.sort(P5[f"tau_{c}"].abs().dropna()); ax[0].plot(v, np.linspace(0, 1, len(v)), col, label=c)
    v = np.sort(P5[f"phase_{c}"].abs().dropna()); ax[1].plot(v, np.linspace(0, 1, len(v)), col)
ax[0].set_xlabel("|time lag| (ms)"); ax[1].set_xlabel("|phase lag| (rad)"); ax[0].legend(); ax[0].set_ylabel("cumulative fraction")
plt.tight_layout(); plt.show()
print("Plotted ranges: |tau| 0 to", P5[["tau_pre", "tau_cooling", "tau_post"]].abs().max().max(), "ms; |phase| 0 to", round(P5[["phase_pre", "phase_cooling", "phase_post"]].abs().max().max(), 3), "rad")

**Result: reproduced.** Over 3136 pairs, time lags are longer during cooling than before
(median 30.5 against 28.5 ms, P = 1.14e-06) and after (P = 1.3e-10), and do not differ between
before and after (P = 0.0967); per session (reading b) the same holds (P = 0.023, 0.016 and
0.81).

## fig5i-phase-lags-unchanged

> Theta phase lags between place cell pairs were not affected by MS cooling (P > 0.24 between all
> conditions).

In [ ]:
for a, b in [("cooling", "pre"), ("cooling", "post"), ("post", "pre")]:
    x, y = P5[f"phase_{a}"].abs(), P5[f"phase_{b}"].abs()
    ok = x.notna() & y.notna()
    p, n = wsr(x[ok], y[ok])
    print(f"|phase lag| {a} against {b}: medians {x[ok].median():.3f} and {y[ok].median():.3f} rad, mean difference {(x[ok] - y[ok]).mean():.4f} rad; signed-rank P = {p:.3g} (n = {n})")
ses_p = P5.assign(**{f"abs_{c}": P5[f"phase_{c}"].abs() for c in ["pre", "cooling", "post"]}).groupby("session")[["abs_pre", "abs_cooling", "abs_post"]].median()
print("variant, session medians:", {f"{a} vs {b}": round(wsr(ses_p[f'abs_{a}'], ses_p[f'abs_{b}'])[0], 3) for a, b in [("cooling", "pre"), ("cooling", "post"), ("post", "pre")]})

**Result: not reproduced.** Phase lags are slightly larger during cooling than before
(median 1.367 against 1.304 rad, mean difference 0.051 rad, P = 0.0015) and after (P = 0.0043),
and before and after do not differ (P = 0.51). The change is small, but by the paper's test the
phase lags are not preserved.

## fig5j-fraction-active-unchanged

> The fraction of place cells active in a theta cycle was not significantly affected by MS cooling
> (P = 0.2).

In [ ]:
CY = pd.concat([units[k]["cycles"].assign(session=k) for k in FIG5], ignore_index=True)
print("Theta cycles during runs in trials:", CY.groupby("state").size().to_dict(), "; place cells per session:", {k: units[k]["n_place_cells"] for k in FIG5})
print("Mean fraction active:", CY.groupby("state").frac_active.mean().round(4).to_dict(), "; median:", CY.groupby("state").frac_active.median().round(4).to_dict())
for a, b in [("pre", "cooling"), ("cooling", "post"), ("pre", "post")]:
    p, na, nb = rks(CY.frac_active[CY.state == a], CY.frac_active[CY.state == b])
    print(f"rank-sum {a} against {b}: P = {p:.3g} (n = {na}, {nb})")
sesj = CY.groupby(["session", "state"]).frac_active.mean().unstack()
print(sesj.round(4).to_string())
print("variant, session means, signed-rank cooling against pre:", round(wsr(sesj.cooling, sesj.pre)[0], 3), "; lower in", int((sesj.cooling < sesj.pre).sum()), "of", len(sesj))

**Result: not reproduced.** Over 9037 pre and 10152 cooling theta cycles, the fraction of
place cells active per cycle is higher during cooling (mean 0.2405 against 0.226, rank-sum
P = 1.5e-22). Per session the cooling mean is lower than pre in 5 of 9 sessions (P = 0.734).

## fig5k-spikes-per-cycle-unchanged

> The number of spikes emitted by all neurons per theta cycle was preserved during MS cooling
> (P > 0.058 between all conditions).

In [ ]:
print("Mean spikes per cycle (all good units):", CY.groupby("state").spikes.mean().round(2).to_dict())
for a, b in [("pre", "cooling"), ("cooling", "post"), ("pre", "post")]:
    p, na, nb = rks(CY.spikes[CY.state == a], CY.spikes[CY.state == b])
    print(f"rank-sum {a} against {b}: P = {p:.3g} (n = {na}, {nb})")
sesk = CY.groupby(["session", "state"]).spikes.mean().unstack()
print(sesk.round(2).to_string())
print("variant, session means, signed-rank cooling against pre:", round(wsr(sesk.cooling, sesk.pre)[0], 3), "; against post:", round(wsr(sesk.cooling, sesk.post)[0], 3))
norm = CY.groupby("session").spikes.transform("mean")
print("variant, spikes per cycle over each session's mean, by state:", (CY.spikes / norm).groupby(CY.state).mean().round(3).to_dict())

**Result: not reproduced.** Spikes per theta cycle do not differ between pre and cooling
(45.68 and 44.98, P = 0.0758) but are higher after cooling (51.68) than during it or before it
(both P below 1e-50), so not all comparisons are null. Per session, cooling differs from neither
pre (P = 0.57) nor post (P = 0.129).

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| session-structure | reproduced |
| cooling-time-course | partially reproduced (minimum after 121 s, not 60 s) |
| fig2g-theta-freq-temperature | reproduced (R = 0.789) |
| fig2h-theta-power-temperature | not reproduced (R = -0.056) |
| fig2j-frequency-consistent-across-animals | reproduced |
| fig2i-theta-pre-cooling-post | partially reproduced (frequency yes, power only against pre) |
| disc-theta-frequency-12-percent | partially reproduced (8.9 to 9.2%) |
| fig6d-errors-increase | partially reproduced (2.3-fold; MS22 not significant) |
| fig6e-errors-recover | reproduced |
| fig6c-worst-trials-4-to-8 | reproduced (trial 7) |
| fig6f-errors-temperature | partially reproduced (R = -0.84, stronger) |
| fig6g-errors-theta-frequency | partially reproduced (R = -0.87, stronger) |
| fig6h-errors-theta-power | partially reproduced (1 rat, not 2) |
| fig6i-errors-not-speed | not reproduced (3 of 5 rats correlate) |
| fig4e-speed-decreases-within-session | reproduced |
| fig5-speed-control-cooling | partially reproduced |
| suppl2-place-cell-rates-decrease | partially reproduced (7.7% and 6.7%) |
| disc-interneuron-rates-decrease | partially reproduced (6.5 to 9.1%) |
| suppl2d-interneuron-oscillation-slows | reproduced |
| suppl1g-waveforms-unchanged | not reproduced (small changes, drift not excluded) |
| fig4b-fp-flfp-decrease | reproduced (all six means) |
| fig4c-fp-flfp-correlation | partially reproduced (weaker R) |
| fig4ac-fp-flfp-difference-preserved | not reproduced (difference falls 0.13 Hz; ratio preserved) |
| fig4a-frequency-temperature-slopes | partially reproduced (3 of 4 values) |
| fig3c-field-size-correlated | reproduced (R = 0.8197, at the edge) |
| fig3d-size-fp-negative | partially reproduced (weaker R) |
| fig3e-model-field-size | partially reproduced (weaker R) |
| fig3f-information-unchanged | not reproduced (small decrease, P = 0.014) |
| fig3hi-population-vector-unchanged | reproduced |
| fig4d-precession-slope-decrease | partially reproduced (one of two comparisons) |
| fig4gh-fp-speed | reproduced |
| fig5d-ccg-period | partially reproduced (lengthens; 109 ms, not 85) |
| fig5cg-compression | not reproduced (0.27 cm/ms; decrease P = 0.055) |
| fig5h-time-lags-increase | reproduced |
| fig5i-phase-lags-unchanged | not reproduced (small increase, P = 0.0015) |
| fig5j-fraction-active-unchanged | not reproduced (pooled cycles differ) |
| fig5k-spikes-per-cycle-unchanged | not reproduced (post differs) |
| suppl1-hippocampal-temperature | blocked: data missing |

How the analysis differs from the paper's. 45 of the paper's 53 sessions have cooling states
here (six are not deposited, one lacks states, one lacks trials); temperature analyses use the 37
with a recorded temperature. Cell types are not in the files and were made here from waveform
width and rate. Place fields use the automatic criteria the Methods give for Figure 5, for all
figures, where the paper defined fields by hand for Figures 3 and 4. Pair lags are the peak of
each pair's correlogram and compression a regression, where the paper fits a surface to the
density of all pairs. The CCG period is measured per pair. Units come from 11 sessions in 4 rats
(the paper's Figure 5 has 18 sessions in 5 rats); MS10 has no curated units. Two unit sessions
were left out because spikes carry no spatial information there.

What was found about the dandiset. The temperature of the nine MS10 sessions and of the two
control sessions of 2018-07-19 is not a recording (the paper's code lists no temperature channel
for them); it runs from exactly 17 to 37 C, or is constant. The two controls also carry cooling
labels although the code describes them as sessions without cooling, and their theta does not
slow as in cooled sessions. Six of the paper's sessions are missing, one session has no cooling
states and one no trials. No file holds cell types, place fields or electrode locations, and the
units of seven sessions are all 'unsorted'. The trial state 'Cooling off' is not described. In
two sessions spikes carry no spatial information. The first published version has no trials table.